# ARCH 6133 Places / Platforms
## NYC Isochrone Lab: Proximity Is Not Access

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danmillr/places-platforms/blob/main/tutorials/ARCH6133_NYC_Isochrone_Lab.ipynb)

---

### What this notebook does

This is a hands-on field guide to **isochrones** — polygons of "everything reachable within X minutes" — and to **walksheds** and **accessibility scores** built on top of them. You will:

- Compare isochrones from three routing APIs (OpenRouteService, Mapbox, Google) for the same origin and see how the answers diverge.
- Compute walksheds at multiple time thresholds, measure their area, network efficiency, and compactness, and detect barriers like highways and waterfronts in the geometry.
- Apply a cumulative-opportunity accessibility score using six NYC Open Data destination layers (parks, schools, libraries, health facilities, retail/grocery, subway entrances).
- Run an explicit equity comparison across five NYC neighborhoods chosen to represent very different positions in the city's transit geography.
- Build your own access question with origins, modes, thresholds, and destination types you choose, and generate an Exploration Card documenting every decision.

This notebook lives in the `tutorials/` folder alongside `ARCH6133_POI_Data.ipynb`, `ARCH6133_Street_View_Lab.ipynb`, `ARCH6133_NYC_Index_Builder.ipynb`, and `ARCH6133_NYC_Site_Selection.ipynb`. It follows the same naming convention.

### How to get the three API keys

**OpenRouteService (free, 2,000 requests/day)**
1. https://openrouteservice.org/dev/#/signup — sign up, confirm email
2. Create a token (default permissions are fine)
3. In Colab Secrets add `ORS_API_KEY` and paste the token

**Mapbox (free tier, 100,000 requests/month)**
1. https://account.mapbox.com — sign up
2. Default public token works for our calls (or create a fresh one with `styles:read` and `isochrone:read` scopes)
3. In Colab Secrets add `MAPBOX_TOKEN`

**Google Maps Platform (the same key from the Street View notebook)**
1. https://console.cloud.google.com → APIs & Services → Library
2. Enable the **Distance Matrix API** in addition to whatever you enabled earlier (Street View Static, Geocoding)
3. In Colab Secrets the key is already named `GOOGLE_API_KEY` if you set it up for the Street View lab
4. Set a **$10 billing cap** under Billing → Budgets & Alerts so you cannot accidentally overspend

If a key is missing, this notebook **does not crash** — it disables the modules that depend on that key and prints a plain-English warning. You can run Modules 0–3 with only the ORS key, Modules 0–4 with ORS + NYC OD (no Mapbox needed), and the full notebook with all three.

### Estimated API call counts per module

| Module | ORS | Mapbox | Google | Notes |
|---|---|---|---|---|
| 0 | 0 | 0 | 1 | One geocoding call to validate the key |
| 1 | 0 | 0 | 0 | Conceptual only |
| 2 | ~15 | ~10 | ~20 | ORS: 5 walking + 5 cycling + 5 driving; Mapbox: 5 walking + 5 cycling; Google: 10-point grid × 2 modes (configurable) |
| 3 | 5 | 0 | 0 | One isochrone per threshold |
| 4 | 0 | 0 | 0 | Uses cached Module 3 walkshed |
| 5 | ~10 | 0 | ~150 | 5 locations × (walking + cycling + 10 destinations × 3 time-of-day variants) |
| 6 | up to 4 | 0 | up to 24 | Depends on student choices |
| 7 | 0 | 0 | 0 | Export only |

**Total at default settings: ~35 ORS calls, ~10 Mapbox calls, ~200 Google Distance Matrix elements.** Comfortably inside ORS's 2,000/day free quota and well under \$1 in Google billing for one full pass.

### Rate limits

| API | Free-tier limit | What the notebook does about it |
|---|---|---|
| ORS | 40 requests/minute, 2,000/day | 1-second sleep between calls |
| Mapbox | 60 requests/minute | 1-second sleep between calls |
| Google | 100 elements/second, billed per element | We batch and use small grids |

### If you want to go deeper

This notebook uses **static** routing APIs, which means it cannot model headway variation across the day or real GTFS schedules (except via Google Distance Matrix, which can). If you need research-grade transit accessibility analysis, look at:

- **[OpenTripPlanner](https://www.opentripplanner.org/)** — the open-source gold standard, ingests GTFS directly
- **[Conveyal Analysis](https://conveyal.com/)** — commercial product on top of OTP for high-resolution access mapping
- **[Accessibility Observatory at the University of Minnesota](https://access.umn.edu/)** — national transit accessibility benchmarks


---

## Module 0 — Setup and Configuration

Run every cell in this module in order. If your runtime resets later, come back here and re-run from the top — Drive mounts, package installs, API keys, and helper functions defined here are used everywhere else.

### Mount Google Drive

Everything this notebook saves — isochrone polygons, accessibility scores, comparison maps, exploration cards — lives in a folder in your Drive.

In [ ]:
# Mount your personal Google Drive into the Colab filesystem.
from google.colab import drive
drive.mount('/content/drive')

### Install required packages

One consolidated install. Takes ~60 seconds the first time. The `openrouteservice` package is the only one not already in Colab; the rest you already have for the other tutorials.

In [ ]:
# One consolidated install (-q quiets dependency-resolution noise).
!pip install -q openrouteservice requests pandas geopandas folium matplotlib seaborn shapely tqdm ipywidgets scikit-learn numpy

### Store all three API keys in Colab Secrets

This notebook uses three routing APIs. Before you run anything below, set up the keys:

| Secret name | Where to get it | What it enables |
|---|---|---|
| `ORS_API_KEY` | https://openrouteservice.org/dev/#/signup (free, 2,000/day) | All ORS isochrones — walking, cycling, driving |
| `MAPBOX_TOKEN` | https://account.mapbox.com (free, 100,000/mo) | Mapbox walking/cycling isochrones for API comparison |
| `GOOGLE_API_KEY` | Reuse the key from `ARCH6133_Street_View_Lab.ipynb`. **Enable the Distance Matrix API** in Google Cloud Console → APIs & Services → Library | Geocoding and transit travel times (only API here with live MTA schedules) |

To add each secret: click the key icon in the left sidebar, click **Add new secret**, paste the value, toggle **Notebook access** ON, and run the next cell.

If a key is missing, the notebook **does not crash** — it will tell you which modules become unavailable and let you continue with the rest.

In [ ]:
# Pull all three keys and validate each one separately.
from google.colab import userdata

ORS_API_KEY    = userdata.get('ORS_API_KEY')
MAPBOX_TOKEN   = userdata.get('MAPBOX_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

missing_keys = []
if not ORS_API_KEY:
    missing_keys.append("ORS_API_KEY")
    print("WARNING: ORS_API_KEY missing. Modules 2, 3, 5, 6 will be unavailable.")
else:
    print("ORS_API_KEY loaded.")

if not MAPBOX_TOKEN:
    missing_keys.append("MAPBOX_TOKEN")
    print("WARNING: MAPBOX_TOKEN missing. Mapbox isochrone comparison in Module 2 "
          "will be skipped.")
else:
    print("MAPBOX_TOKEN loaded.")

if not GOOGLE_API_KEY:
    missing_keys.append("GOOGLE_API_KEY")
    print("WARNING: GOOGLE_API_KEY missing. Geocoding and Google Distance Matrix "
          "calls will be unavailable. You will need to provide coordinate pairs "
          "directly instead of addresses.")
else:
    print("GOOGLE_API_KEY loaded.")

if missing_keys:
    print(f"\n{len(missing_keys)} key(s) missing: {missing_keys}")
    print("The notebook will continue — affected modules will print a clear notice.")
else:
    print("\nAll three API keys loaded successfully.")

### Configure outputs

Edit these values to change the study area or time thresholds. Defaults are tuned for an East Harlem / Lower Manhattan teaching workflow.

In [ ]:
# Project configuration. Edit and re-run if you change defaults.
OUTPUT_FOLDER = '/content/drive/MyDrive/NYCIsochroneLab/'

ORIGIN_ADDRESS = '175 Water St, New York, NY 10038'

COMPARISON_ADDRESSES = [
    'Times Square, New York, NY',
    '165 Rockaway Ave, Brooklyn, NY 11233',
    '149th St and Grand Concourse, Bronx, NY 10451',
    'Jamaica Center, Queens, NY 11435',
    'St George Ferry Terminal, Staten Island, NY 10301',
]

TIME_THRESHOLDS = [5, 10, 15, 20, 30]   # minutes
TOP_N = 10

# Google Distance Matrix is billed per element. The Module 2 spot-check grid is
# configurable so you can keep costs predictable. 10 points * 2 modes = 20
# elements per full pass (~$0.10 at current rates). Raise to 20 for tighter
# validation if your billing cap allows.
GOOGLE_GRID_POINTS = 10

# ORS foot-walking maxes out at 60 minutes per request. The notebook caps any
# threshold above 60 with a printed warning rather than letting ORS silently
# truncate.
ORS_MAX_MINUTES = 60

import os
SUBFOLDERS = ['isochrones', 'walksheds', 'accessibility', 'comparison',
              'timeofday', 'exports', 'maps']
for sub in [''] + SUBFOLDERS:
    os.makedirs(os.path.join(OUTPUT_FOLDER, sub), exist_ok=True)

# Enable ipywidgets in Colab.
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()

# Warn if any thresholds exceed the ORS foot-walking limit.
over_limit = [m for m in TIME_THRESHOLDS if m > ORS_MAX_MINUTES]
if over_limit:
    print(f"WARNING: TIME_THRESHOLDS contains values above the ORS 60-minute cap: "
          f"{over_limit}. These will be dropped from ORS calls and you will only "
          f"see Mapbox / Google results for them.")

print(f"\nSetup complete. Output folder ready at {OUTPUT_FOLDER}.")
for sub in SUBFOLDERS:
    print(f"  - {sub}/")

### Reusable helper functions

Five helpers every later module reuses. Each has a one-line docstring; read the source before you use them if you want to know exactly what is wrapped.

In [ ]:
# Reusable helpers. All five have one-line docstrings.
import time
import math
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape, Point, Polygon
from tqdm.auto import tqdm

def geocode_address(address, api_key):
    """Convert a street address to (lat, lng) via Google Geocoding API."""
    if not api_key:
        print(f"  Cannot geocode '{address}': no GOOGLE_API_KEY.")
        return None
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": address, "key": api_key}
    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()
    except Exception as fetch_error:
        print(f"  Geocoding network failure for '{address}': {fetch_error}")
        return None
    if data.get("status") != "OK" or not data.get("results"):
        print(f"  Geocoder could not resolve '{address}'. Status: {data.get('status')}")
        return None
    loc = data["results"][0]["geometry"]["location"]
    return (loc["lat"], loc["lng"])

def make_ors_isochrone(coords, profile, minutes, api_key):
    """Return a shapely Polygon for an ORS isochrone at (lng,lat), profile, minutes."""
    if not api_key:
        return None
    if minutes > ORS_MAX_MINUTES and profile == "foot-walking":
        print(f"  Skipping ORS {profile} at {minutes} min (over {ORS_MAX_MINUTES}-min cap).")
        return None
    url = f"https://api.openrouteservice.org/v2/isochrones/{profile}"
    headers = {"Authorization": api_key,
               "Content-Type": "application/json; charset=utf-8"}
    body = {"locations": [list(coords)],
            "range": [int(minutes) * 60],
            "range_type": "time"}
    try:
        response = requests.post(url, json=body, headers=headers, timeout=30)
        response.raise_for_status()
        payload = response.json()
    except Exception as call_error:
        print(f"  ORS isochrone failed ({profile}, {minutes} min): {call_error}")
        return None
    try:
        geometry = shape(payload["features"][0]["geometry"])
        return geometry
    except Exception as parse_error:
        print(f"  ORS response could not be parsed: {parse_error}")
        return None

def make_mapbox_isochrone(coords, profile, minutes, token):
    """Return a shapely Polygon for a Mapbox isochrone at (lng,lat), profile, minutes."""
    if not token:
        return None
    lng, lat = coords
    url = (f"https://api.mapbox.com/isochrone/v1/mapbox/{profile}/"
           f"{lng},{lat}")
    params = {"contours_minutes": str(int(minutes)),
              "polygons": "true",
              "access_token": token}
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        payload = response.json()
    except Exception as call_error:
        print(f"  Mapbox isochrone failed ({profile}, {minutes} min): {call_error}")
        return None
    try:
        geometry = shape(payload["features"][0]["geometry"])
        return geometry
    except Exception as parse_error:
        print(f"  Mapbox response could not be parsed: {parse_error}")
        return None

def google_distance_matrix(origins, destinations, mode, api_key,
                           departure_time=None):
    """Return a DataFrame of travel times in minutes from origins to destinations."""
    if not api_key:
        return pd.DataFrame()
    url = "https://maps.googleapis.com/maps/api/distancematrix/json"
    def _coord_string(point):
        if isinstance(point, str):
            return point
        return f"{point[0]},{point[1]}"
    params = {"origins":      "|".join(_coord_string(o) for o in origins),
              "destinations": "|".join(_coord_string(d) for d in destinations),
              "mode":         mode,
              "key":          api_key}
    if departure_time is not None:
        params["departure_time"] = str(int(departure_time))
    try:
        response = requests.get(url, params=params, timeout=30)
        data = response.json()
    except Exception as call_error:
        print(f"  Google Distance Matrix call failed: {call_error}")
        return pd.DataFrame()
    if data.get("status") != "OK":
        print(f"  Google Distance Matrix non-OK status: {data.get('status')}")
        return pd.DataFrame()
    rows = []
    for origin_index, row in enumerate(data.get("rows", [])):
        for destination_index, element in enumerate(row.get("elements", [])):
            if element.get("status") != "OK":
                travel_minutes = None
            else:
                # Use duration_in_traffic when available (driving + departure_time);
                # for transit and walking, plain duration is what we want.
                duration_seconds = (element.get("duration_in_traffic") or
                                    element.get("duration") or {}).get("value")
                travel_minutes = (duration_seconds / 60.0
                                  if duration_seconds is not None else None)
            rows.append({"origin_index":      origin_index,
                         "destination_index": destination_index,
                         "travel_minutes":    travel_minutes,
                         "status":            element.get("status")})
    return pd.DataFrame(rows)

def normalize_minmax(series):
    """Rescale a numeric Series to [0, 1] using its observed min and max."""
    numeric = pd.to_numeric(series, errors="coerce")
    minimum = numeric.min(skipna=True)
    maximum = numeric.max(skipna=True)
    if pd.isna(minimum) or pd.isna(maximum) or maximum == minimum:
        return pd.Series(np.zeros(len(numeric)), index=series.index)
    return (numeric - minimum) / (maximum - minimum)

print("Helpers ready: geocode_address, make_ors_isochrone, make_mapbox_isochrone, "
      "google_distance_matrix, normalize_minmax.")

### Test the geocoder

Quick check that your Google key works for geocoding before any expensive routing calls.

In [ ]:
# Geocode the configured origin so we have working coordinates downstream.
ORIGIN_LATLNG = geocode_address(ORIGIN_ADDRESS, GOOGLE_API_KEY)
if ORIGIN_LATLNG is None:
    print(f"Could not geocode {ORIGIN_ADDRESS}. Check your GOOGLE_API_KEY or edit the address.")
else:
    print(f"Origin geocoded: {ORIGIN_ADDRESS}")
    print(f"  latitude/longitude: {ORIGIN_LATLNG}")
print()
print("Module 0 setup complete.")

---

## Module 1 — Conceptual Foundation: How Isochrones Work

This module produces **no API calls** beyond geocoding. It is the conceptual ground every later module stands on. Read it carefully — if you skip to Module 2 you will be drawing maps without understanding what they claim.

### Proximity is not access

A 15-minute isochrone drawn from a point in Midtown Manhattan and a 15-minute isochrone drawn from a point in East New York cover radically different amounts of the city, reach radically different destinations, and represent radically different lived experiences of urban mobility. **The isochrone is a tool for making that difference visible — but only if we ask the right questions of it.**

Isochrone analysis as a contemporary urban-design tool got most of its current visibility from the **15-minute city** concept, formalized by Carlos Moreno (Sorbonne) around 2016 and adopted as official policy in **Paris** under Anne Hidalgo, then echoed in **Portland**, **Melbourne**, **Bogotá**, and a long list of other cities. The idea is simple and politically resonant: every resident should be able to reach the essentials of daily life — work, school, food, health care, parks — within a 15-minute walk or bike ride from home. The critics are sharp: the 15-minute city works well in dense, mixed-use European city centers; it works less well in American cities organized around car ownership and concentrated disinvestment; and at its weakest moments it can rhetorically substitute spatial proximity for the harder politics of housing, wages, and service quality.

This notebook does not litigate the 15-minute city. It does something simpler and more useful: it makes the **geometry of access** measurable so you can see where the rhetoric lines up with the city as it actually is, and where it does not.

### A routing API knows about roads and schedules

A routing API knows about roads and schedules. **It does not know about the woman who will not walk past that corner after dark, the man whose wheelchair cannot navigate that curb cut, or the family that cannot afford the subway fare.** Keep that gap in mind as you read every map in this notebook.

### The three geometries of access

**Euclidean buffer.** A circle of radius `r = speed × time`. Fast, simple, wrong. Assumes you can walk through buildings, across rivers, over highway sound walls, and through fenced-off rail yards. It is useful as a rough planning sketch and as a benchmark for measuring how much the actual street network slows you down — but it does not describe how anyone actually moves.

**Network isochrone.** Follows the actual street or transit network as a graph. This is what OpenRouteService, Mapbox, and Google produce. It reveals how block structure, dead-end streets, highway barriers, and transit lines shape the geography of access. Three different routing engines will give you three different network isochrones for the same trip — see Module 2 for a side-by-side.

**True walkshed.** A network isochrone restricted to **pedestrian** infrastructure — sidewalks, crosswalks, pedestrian bridges. Reveals gaps in the pedestrian network that a driving or general-network isochrone would not show. ORS's `foot-walking` profile approximates this, but OpenStreetMap (its underlying data) has known gaps in pedestrian-infrastructure tagging in NYC's outer boroughs. Module 3 dives into this.

### How a routing API works, briefly

Routing APIs model the street network as a **graph**: intersections are **nodes**, street segments are **edges**, and each edge has a **weight** representing its travel cost (time, distance, or an impedance like "avoid highways"). To compute an isochrone, the API runs **Dijkstra's algorithm** or a similar shortest-path procedure from your origin: it fans out across the network, always extending the cheapest known path first, until it has found the travel time from your origin to every reachable node. **The isochrone polygon is the boundary of all nodes reachable within your time threshold.**

Transit routing adds **schedule data** on top of the network graph. The algorithm now has to know when the next bus or train leaves, how long the transfer takes, and whether the route runs on a weekend or at 9pm. That schedule information lives in a standard called **GTFS**.

### GTFS, briefly

If you want to go deeper into transit routing, **GTFS (General Transit Feed Specification)** is the foundation. It is a standard format for publishing transit schedules used by every major transit agency in the United States. The MTA publishes its full GTFS feed at [transitfeeds.com](https://transitfeeds.com/) and on the [MTA Developer Resources](https://api.mta.info/) page.

Tools like **[OpenTripPlanner (OTP)](https://www.opentripplanner.org/)** and **[Conveyal Analysis](https://conveyal.com/)** ingest GTFS directly to produce highly accurate transit isochrones that account for real schedules, headways, and transfers. OTP is the gold standard for research-grade transit access analysis but requires standing up a server — it is beyond the scope of this notebook but worth knowing exists.

In this notebook, **Google Distance Matrix is our GTFS-aware transit engine**: it ingests MTA's GTFS feed and returns scheduled travel times that vary by time of day. ORS and Mapbox use static street network data and cannot model transit headways at all. That is one of the most important practical differences between free and commercial routing APIs.

### Illustrative figure: three geometries, same origin

The figure below is **schematic** — it does not call any API. It shows the three geometries side by side at the same scale, for the same imaginary origin, so you can read them as competing claims about where you can go in fifteen minutes.

In [ ]:
# Module 1 — conceptual figure. No API calls.
import os
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon, Circle as MplCircle

# Imaginary origin at the origin (0, 0).
WALKING_SPEED_M_PER_MIN = 80  # ~ 4.8 km/h, a planning standard
minutes = 15
radius_m = WALKING_SPEED_M_PER_MIN * minutes

fig, ax = plt.subplots(figsize=(9, 9))

# 1. Euclidean circle.
ax.add_patch(MplCircle((0, 0), radius_m, fill=False,
                       linestyle="--", linewidth=2, edgecolor="#999999",
                       label=f"Euclidean buffer ({radius_m} m)"))

# 2. Schematic network isochrone — hand-coded irregular polygon shaped
#    like an asymmetric blob that conforms loosely to a grid.
network_vertices = np.array([
    [ 950,    50], [ 850,   350], [ 600,   600], [ 350,   700], [   0,   800],
    [-350,   700], [-700,   550], [-900,   300], [-980,     0], [-900,  -300],
    [-700,  -550], [-350,  -700], [   0,  -780], [ 350,  -700], [ 650,  -550],
    [ 870,  -300], [ 950,   -50],
])
ax.add_patch(MplPolygon(network_vertices, closed=True, fill=True,
                        facecolor="#3C4ED6", edgecolor="#1B1B33", alpha=0.35,
                        linewidth=1.5, label="Network isochrone (schematic)"))

# 3. Schematic walkshed — same blob but with a notch cut where a barrier
#    (highway or river) would block pedestrian access on the east side.
walkshed_vertices = np.array([
    [ 700,    50], [ 650,   200], [ 600,   600], [ 350,   700], [   0,   800],
    [-350,   700], [-700,   550], [-900,   300], [-980,     0], [-900,  -300],
    [-700,  -550], [-350,  -700], [   0,  -780], [ 350,  -700], [ 600,  -550],
    [ 700,  -300], [ 700,  -100],
])
ax.add_patch(MplPolygon(walkshed_vertices, closed=True, fill=True,
                        facecolor="#E67E22", edgecolor="#7B241C", alpha=0.55,
                        linewidth=1.5, label="True walkshed (schematic, with barrier)"))

# Origin marker.
ax.plot(0, 0, "o", color="#1B1B33", markersize=10, zorder=5)
ax.annotate("Origin", xy=(0, 0), xytext=(50, 60),
            fontsize=11, fontweight="bold")

# Barrier hint.
ax.annotate("(schematic barrier:\nhighway or river\nblocks pedestrians)",
            xy=(900, 0), xytext=(1050, -50),
            fontsize=9, color="#7B241C")

ax.set_xlim(-1500, 1500); ax.set_ylim(-1500, 1500)
ax.set_aspect("equal")
ax.axhline(0, color="#cccccc", linewidth=0.5)
ax.axvline(0, color="#cccccc", linewidth=0.5)
ax.set_xlabel("meters east of origin")
ax.set_ylabel("meters north of origin")
ax.set_title(f"{minutes}-minute walking access — three geometries, one origin")
ax.legend(loc="lower left")
caption = "Same origin. Same time. Three different claims about where you can go."
fig.text(0.5, 0.01, caption, ha="center", fontsize=10, style="italic")
plt.tight_layout(rect=[0, 0.03, 1, 1])

out_path = os.path.join(OUTPUT_FOLDER, "isochrones", "conceptual_figure.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")

**What this analysis cannot tell you.**

- The Euclidean circle ignores all infrastructure — useful only as a benchmark, never as a real claim about access.
- The network isochrone assumes the routing engine's underlying street data is accurate; for NYC, OSM is excellent in Manhattan and patchy in outer-borough cul-de-sacs and superblock interiors.
- The walkshed shape depends on which sidewalks and crosswalks are *tagged* in OSM — missing pedestrian infrastructure data shows up as an isochrone that follows car streets instead of pedestrian routes.

---

## Module 2 — Single Origin Comparative Isochrones

We now fetch real isochrones from three different APIs for one origin, overlay them, and measure how much they disagree. This is the core comparative module: it shows you what "the answer" looks like when there is no single answer.

**Module 2 depends on:** Module 0 (helpers, keys). Will skip gracefully if any of the three keys are missing.

### A note on the three APIs

**[OpenRouteService](https://openrouteservice.org/)** is open-source, built on **OpenStreetMap** data, free for 2,000 requests/day. The data is community-maintained — which means it improves continuously, and means it varies in completeness by neighborhood. The routing engine and the underlying data are both inspectable; you can read the source code, file an issue, or fix a missing sidewalk yourself.

**[Mapbox](https://www.mapbox.com/)** is commercial. Its street data is proprietary (a hybrid of OSM, commercial feeds, and Mapbox-collected drive traces), generally more current in some areas, and has a notably better walking-network model in some US cities. It is a black box — you cannot see why Mapbox routed you one way and not another.

**[Google Maps Platform](https://mapsplatform.google.com/)** is also commercial. For NYC transit specifically, it is the most accurate of the three because it ingests **live MTA GTFS data** and adjusts for scheduled headway, transfers, and time of day. It is expensive at scale (billed per element) and a complete black box.

**When three APIs give you three different answers for the same trip, that is not an error — it is information about the uncertainty in any routing model.**

ORS and Mapbox both **do not support transit isochrones** in their free/standard tiers (the open-source routing layer does not include a GTFS-aware engine). Google does. That structural difference is why we use Google for the spot-check in this module and for all transit calls later.

In [ ]:
# Module 2 — fetch and overlay isochrones from ORS and Mapbox.
import os
import json
import time
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import shape, Point, Polygon
from tqdm.auto import tqdm

if ORIGIN_LATLNG is None:
    print("No origin coordinates available. Run the geocoder cell in Module 0 first.")
    raise SystemExit

origin_lat, origin_lng = ORIGIN_LATLNG
origin_coords = (origin_lng, origin_lat)  # ORS / Mapbox both expect (lng, lat)

# Storage for everything we fetch in this module.
ors_results    = {"foot-walking": {}, "cycling-regular": {}, "driving-car": {}}
mapbox_results = {"walking": {},      "cycling": {}}

# ORS — three modes × every threshold.
if ORS_API_KEY:
    print("Fetching ORS isochrones...")
    ors_eligible = [m for m in TIME_THRESHOLDS if m <= ORS_MAX_MINUTES]
    for profile_name in ["foot-walking", "cycling-regular", "driving-car"]:
        for minutes in tqdm(ors_eligible, desc=f"ORS {profile_name}"):
            polygon = make_ors_isochrone(origin_coords, profile_name,
                                         minutes, ORS_API_KEY)
            if polygon is not None:
                ors_results[profile_name][minutes] = polygon
            time.sleep(1.0)  # ORS free tier: 40 req/min
else:
    print("ORS_API_KEY missing — skipping all ORS isochrone fetches.")

# Mapbox — two modes × every threshold.
if MAPBOX_TOKEN:
    print("\nFetching Mapbox isochrones...")
    for profile_name in ["walking", "cycling"]:
        for minutes in tqdm(TIME_THRESHOLDS, desc=f"Mapbox {profile_name}"):
            polygon = make_mapbox_isochrone(origin_coords, profile_name,
                                            minutes, MAPBOX_TOKEN)
            if polygon is not None:
                mapbox_results[profile_name][minutes] = polygon
            time.sleep(1.0)  # Mapbox free tier: 60 req/min
else:
    print("MAPBOX_TOKEN missing — Mapbox isochrones will not be drawn.")

print(f"\nORS isochrones fetched : "
      f"{sum(len(v) for v in ors_results.values())}")
print(f"Mapbox isochrones fetched: "
      f"{sum(len(v) for v in mapbox_results.values())}")

In [ ]:
# Save raw isochrone polygons to disk as GeoJSON for downstream modules.
def _polygons_to_geojson(results_by_profile, source_label):
    """Turn a nested {profile: {minutes: polygon}} dict into a FeatureCollection."""
    features = []
    for profile_name, by_minutes in results_by_profile.items():
        for minutes, polygon in by_minutes.items():
            if polygon is None:
                continue
            features.append({
                "type": "Feature",
                "geometry": polygon.__geo_interface__,
                "properties": {"source":  source_label,
                               "profile": profile_name,
                               "minutes": int(minutes)},
            })
    return {"type": "FeatureCollection", "features": features}

ors_geojson    = _polygons_to_geojson(ors_results,    "ors")
mapbox_geojson = _polygons_to_geojson(mapbox_results, "mapbox")

ors_path    = os.path.join(OUTPUT_FOLDER, "isochrones", "ors_raw.geojson")
mapbox_path = os.path.join(OUTPUT_FOLDER, "isochrones", "mapbox_raw.geojson")
with open(ors_path,    "w") as out_file: json.dump(ors_geojson,    out_file)
with open(mapbox_path, "w") as out_file: json.dump(mapbox_geojson, out_file)
print(f"Saved: {ors_path}")
print(f"Saved: {mapbox_path}")

### Google Distance Matrix spot-check

Google does not return isochrone polygons natively. To construct one you would need to call the Distance Matrix API from a grid of destination points and interpolate the boundary — expensive and slow at scale.

Instead we use Google here as a **point-by-point validator**: pick a grid of destination points sampled from inside the ORS 15-minute walking isochrone, ask Google how long it takes to walk and to take transit from the origin to each of them, and check whether Google's walking times agree with the ORS isochrone's claim that those points are reachable in 15 minutes.

The grid size is set by `GOOGLE_GRID_POINTS` in the config cell (default 10). At Google's current Distance Matrix pricing this is ~\$0.10 per full run.

In [ ]:
# Build a random sample of N destination points inside the 15-min ORS walking
# isochrone, then call Google Distance Matrix for walking and transit times.
import random
random.seed(42)

if not GOOGLE_API_KEY:
    print("GOOGLE_API_KEY missing — skipping the spot-check.")
    google_spot_check_df = pd.DataFrame()
elif 15 not in ors_results.get("foot-walking", {}):
    print("No 15-min ORS walking isochrone found. Re-run the ORS cell first.")
    google_spot_check_df = pd.DataFrame()
else:
    isochrone_polygon = ors_results["foot-walking"][15]
    minx, miny, maxx, maxy = isochrone_polygon.bounds
    grid_points = []
    attempts = 0
    # Rejection-sample uniformly inside the polygon's bounding box.
    while len(grid_points) < GOOGLE_GRID_POINTS and attempts < 1000:
        candidate_lng = random.uniform(minx, maxx)
        candidate_lat = random.uniform(miny, maxy)
        if isochrone_polygon.contains(Point(candidate_lng, candidate_lat)):
            grid_points.append((candidate_lat, candidate_lng))
        attempts += 1
    print(f"Sampled {len(grid_points)} grid points inside the 15-min ORS walkshed.")

    # Walking call (no departure_time).
    walking_df = google_distance_matrix(
        origins=[ORIGIN_LATLNG], destinations=grid_points,
        mode="walking", api_key=GOOGLE_API_KEY)
    # Transit call — Google requires a future departure_time.
    next_weekday_8am = pd.Timestamp.utcnow().normalize() + pd.Timedelta(days=1)
    while next_weekday_8am.dayofweek >= 5:
        next_weekday_8am += pd.Timedelta(days=1)
    next_weekday_8am = (next_weekday_8am.replace(hour=12).timestamp())
    transit_df = google_distance_matrix(
        origins=[ORIGIN_LATLNG], destinations=grid_points,
        mode="transit", api_key=GOOGLE_API_KEY,
        departure_time=next_weekday_8am)

    rows = []
    for index, (lat, lng) in enumerate(grid_points):
        walk_min = (walking_df.query(f"destination_index == {index}")
                    ["travel_minutes"].iloc[0] if not walking_df.empty else None)
        transit_min = (transit_df.query(f"destination_index == {index}")
                       ["travel_minutes"].iloc[0] if not transit_df.empty else None)
        rows.append({"point_id": index, "lat": lat, "lng": lng,
                     "google_walk_min":    walk_min,
                     "google_transit_min": transit_min})
    google_spot_check_df = pd.DataFrame(rows)

    print()
    print("Google times for points sampled inside the ORS 15-min walking isochrone:")
    print("(if ORS and Google agree, walking times should mostly fall below 15)")
    display(google_spot_check_df)
    over_15 = (google_spot_check_df["google_walk_min"] > 15).sum()
    print(f"\nPoints where Google walking > 15 min: {over_15} / "
          f"{len(google_spot_check_df)}  (high values mean ORS over-claimed)")

    out_csv = os.path.join(OUTPUT_FOLDER, "isochrones", "google_spot_check.csv")
    google_spot_check_df.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

In [ ]:
# Folium visualization: ORS + Mapbox isochrones overlaid for each mode.
center = [origin_lat, origin_lng]
comparative_map = folium.Map(location=center, zoom_start=14, tiles="cartodbpositron")

# Color scale: deeper saturation with increasing time.
def _shade(base_rgb, fraction):
    """Return a hex color interpolated between white and base_rgb at fraction."""
    interpolated = tuple(int(255 - (255 - channel) * fraction)
                         for channel in base_rgb)
    return "#%02x%02x%02x" % interpolated

# Walking layer group — ORS (blues) + Mapbox (greens).
walking_group = folium.FeatureGroup(name="Walking — ORS vs Mapbox", show=True)
for index, minutes in enumerate(sorted(ors_results.get("foot-walking", {}).keys())):
    fraction = (index + 1) / max(len(ors_results["foot-walking"]), 1)
    color = _shade((60, 78, 214), fraction)  # ORS blue family
    folium.GeoJson(ors_results["foot-walking"][minutes].__geo_interface__,
                   style_function=lambda feature, c=color: {"fillColor": c,
                                                             "color": c,
                                                             "weight": 1.5,
                                                             "fillOpacity": 0.18},
                   tooltip=f"ORS walking — {minutes} min").add_to(walking_group)
for index, minutes in enumerate(sorted(mapbox_results.get("walking", {}).keys())):
    fraction = (index + 1) / max(len(mapbox_results["walking"]), 1)
    color = _shade((22, 160, 133), fraction)  # Mapbox green family
    folium.GeoJson(mapbox_results["walking"][minutes].__geo_interface__,
                   style_function=lambda feature, c=color: {"fillColor": c,
                                                             "color": c,
                                                             "weight": 1.5,
                                                             "fillOpacity": 0.18,
                                                             "dashArray": "4, 3"},
                   tooltip=f"Mapbox walking — {minutes} min").add_to(walking_group)
walking_group.add_to(comparative_map)

# Cycling layer group.
cycling_group = folium.FeatureGroup(name="Cycling — ORS vs Mapbox", show=False)
for index, minutes in enumerate(sorted(ors_results.get("cycling-regular", {}).keys())):
    fraction = (index + 1) / max(len(ors_results["cycling-regular"]), 1)
    color = _shade((60, 78, 214), fraction)
    folium.GeoJson(ors_results["cycling-regular"][minutes].__geo_interface__,
                   style_function=lambda feature, c=color: {"fillColor": c,
                                                             "color": c,
                                                             "weight": 1.5,
                                                             "fillOpacity": 0.18},
                   tooltip=f"ORS cycling — {minutes} min").add_to(cycling_group)
for index, minutes in enumerate(sorted(mapbox_results.get("cycling", {}).keys())):
    fraction = (index + 1) / max(len(mapbox_results["cycling"]), 1)
    color = _shade((22, 160, 133), fraction)
    folium.GeoJson(mapbox_results["cycling"][minutes].__geo_interface__,
                   style_function=lambda feature, c=color: {"fillColor": c,
                                                             "color": c,
                                                             "weight": 1.5,
                                                             "fillOpacity": 0.18,
                                                             "dashArray": "4, 3"},
                   tooltip=f"Mapbox cycling — {minutes} min").add_to(cycling_group)
cycling_group.add_to(comparative_map)

# Driving layer group (ORS only).
driving_group = folium.FeatureGroup(name="Driving — ORS only", show=False)
for index, minutes in enumerate(sorted(ors_results.get("driving-car", {}).keys())):
    fraction = (index + 1) / max(len(ors_results["driving-car"]), 1)
    color = _shade((123, 36, 28), fraction)
    folium.GeoJson(ors_results["driving-car"][minutes].__geo_interface__,
                   style_function=lambda feature, c=color: {"fillColor": c,
                                                             "color": c,
                                                             "weight": 1.5,
                                                             "fillOpacity": 0.15},
                   tooltip=f"ORS driving — {minutes} min").add_to(driving_group)
driving_group.add_to(comparative_map)

# Origin marker.
folium.Marker(location=[origin_lat, origin_lng],
              popup=ORIGIN_ADDRESS,
              icon=folium.Icon(color="darkblue", icon="home")).add_to(comparative_map)

# Legend (manual HTML — folium does not have a first-class legend widget).
legend_html = '''
<div style="position: fixed; bottom: 20px; left: 20px; width: 230px;
            background: white; padding: 8px 10px; border: 1px solid #999;
            font-size: 12px; line-height: 1.4;">
<b>Comparative isochrones</b><br>
<span style="color:#3C4ED6">&#9632;</span> ORS (blue, solid)<br>
<span style="color:#16A085">&#9632;</span> Mapbox (green, dashed)<br>
<span style="color:#7B241C">&#9632;</span> ORS driving (red, separate layer)<br>
Lighter = shorter time; darker = longer time
</div>'''
comparative_map.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(comparative_map)

map_path = os.path.join(OUTPUT_FOLDER, "maps", "comparative_isochrones.html")
comparative_map.save(map_path)
print(f"Saved: {map_path}")
display(comparative_map)

In [ ]:
# Summary table — area, perimeter, and network efficiency at the 15-min walking
# threshold for ORS and Mapbox. All math done in EPSG:2263 (NYC State Plane, feet)
# for accurate area and length.
WALKING_SPEED_M_PER_MIN = 80.0
SQFT_PER_SQKM = 10763910.41671  # 1 km^2 in sqft
FT_PER_KM     = 3280.84

def _summary_row(polygon, source_label, minutes):
    """Return one row of metrics for a single isochrone polygon."""
    if polygon is None:
        return None
    geoseries = gpd.GeoSeries([polygon], crs="EPSG:4326").to_crs(epsg=2263)
    area_sqkm = float(geoseries.geometry.area.iloc[0]) / SQFT_PER_SQKM
    perimeter_km = float(geoseries.geometry.length.iloc[0]) / FT_PER_KM
    radius_m = WALKING_SPEED_M_PER_MIN * minutes
    euclidean_area_sqkm = math.pi * (radius_m / 1000.0) ** 2
    efficiency = (area_sqkm / euclidean_area_sqkm
                  if euclidean_area_sqkm > 0 else float('nan'))
    return {"source": source_label, "minutes": minutes,
            "area_sqkm":           round(area_sqkm, 3),
            "perimeter_km":        round(perimeter_km, 2),
            "euclidean_area_sqkm": round(euclidean_area_sqkm, 3),
            "network_efficiency_ratio": round(efficiency, 3)}

summary_rows = []
if ors_results.get("foot-walking", {}).get(15) is not None:
    summary_rows.append(_summary_row(ors_results["foot-walking"][15], "ORS",    15))
if mapbox_results.get("walking", {}).get(15) is not None:
    summary_rows.append(_summary_row(mapbox_results["walking"][15], "Mapbox", 15))
summary_df = pd.DataFrame(summary_rows)
print("15-minute walking isochrone metrics:")
display(summary_df)

**Network efficiency, read out loud.** A network efficiency ratio of 0.6 means: in 15 minutes of walking, you can actually reach only 60% of the area a perfect omni-directional walker could reach if streets ran in all directions. **In Manhattan's grid this ratio is usually high — close to 0.7 or 0.8.** In neighborhoods fragmented by highways, rail yards, or waterfronts (Red Hook, Hunts Point, Roosevelt Island, Mott Haven) it can drop **below 0.4**. That number is the geometry of structural disinvestment, made measurable.

**What this analysis cannot tell you.**

- The two APIs use different underlying street data. Where they agree, you have a strong signal. Where they disagree (especially in outer-borough industrial areas), neither is necessarily right — both may be wrong in different ways.
- ORS does not support transit; Mapbox does not either. The walking isochrones above are not a substitute for a transit access map.
- The spot-check above tells you whether *the APIs agree* with each other — it does not tell you whether either matches the actual time a real person would take, which depends on traffic signals, sidewalk congestion, weather, and ability.

---

## Module 3 — Walkshed Analysis

This module focuses on **pedestrian** access specifically, using ORS `foot-walking` isochrones, and introduces pedestrian-network barriers as a measurable spatial-equity issue.

**Module 3 depends on:** Module 0 (ORS key). Re-fetches its own isochrones so it runs independently after a runtime reset.

### What a walkshed is

A **walkshed** is the area you can reach on foot within a time threshold, following pedestrian infrastructure only — sidewalks, crosswalks, pedestrian bridges. It is the fundamental unit of:

- The **15-minute city** concept of daily-life access
- The **transit catchment area** (everywhere within walking distance of a station)
- **School siting** radius analyses
- The **service area** of a public library, clinic, or community center

For people who do not own cars, the walkshed *is* their geography of everyday life.

### Barriers that distort walksheds in NYC

- **Waterfront expressways** — the FDR Drive on Manhattan's east side, the West Side Highway, the Belt Parkway around Brooklyn and Queens — physically cut off the waterfront from the inland street network for pedestrians.
- **Rail yards** — Hudson Yards (pre-redevelopment), Sunnyside Yard, the Bedford-Stuyvesant rail cut, the Long Island Rail Road right-of-way — create wide voids where pedestrians cannot cross.
- **Highway sound walls** — the BQE through Brooklyn Heights, Carroll Gardens, and Sunset Park has sound walls that block pedestrian crossings for blocks at a time.
- **Missing sidewalks** — outer-borough residential neighborhoods (parts of Eastern Queens, Staten Island, the South Bronx) have streets without continuous sidewalks. OSM's coverage of these gaps is improving but incomplete.
- **Superblock dead-ends** — large NYCHA developments and 20th-century housing projects sometimes have interior pathways that are pedestrian-accessible but not always tagged as such in OSM.

OpenStreetMap (which ORS uses) is generally excellent for Manhattan's street network and good for the dense parts of Brooklyn and Queens. Pedestrian-infrastructure tagging in outer-borough residential areas is the known weakness. **A walkshed that looks tight or fragmented in those areas may be a real signal of poor pedestrian infrastructure, or it may be a data-quality artifact.** Often it is both.

In [ ]:
# Module 3 — fetch walksheds, measure their geometry, detect barriers.
import os
import math
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
from shapely.geometry import Point
from tqdm.auto import tqdm

if not ORS_API_KEY:
    print("ORS_API_KEY missing — cannot run Module 3.")
    raise SystemExit
if ORIGIN_LATLNG is None:
    print("No origin — re-run Module 0 first.")
    raise SystemExit

origin_lat, origin_lng = ORIGIN_LATLNG
origin_coords = (origin_lng, origin_lat)

print("Fetching ORS foot-walking isochrones for the walkshed analysis...")
walkshed_polygons = {}
for minutes in tqdm(TIME_THRESHOLDS, desc="walksheds"):
    if minutes > ORS_MAX_MINUTES:
        continue
    polygon = make_ors_isochrone(origin_coords, "foot-walking",
                                 minutes, ORS_API_KEY)
    if polygon is not None:
        walkshed_polygons[minutes] = polygon
    import time as _t; _t.sleep(1.0)

if not walkshed_polygons:
    print("No walksheds fetched — ORS may be rate-limited or down. "
          "Try again in a minute.")
    raise SystemExit

print(f"Fetched {len(walkshed_polygons)} walksheds.")

In [ ]:
# Walkshed metrics — area, efficiency, perimeter, compactness — per threshold.
SQFT_PER_SQKM = 10763910.41671
SQFT_PER_ACRE = 43560.0
FT_PER_KM     = 3280.84
WALKING_SPEED_M_PER_MIN = 80.0

metric_rows = []
for minutes in sorted(walkshed_polygons.keys()):
    polygon = walkshed_polygons[minutes]
    projected = gpd.GeoSeries([polygon], crs="EPSG:4326").to_crs(epsg=2263).iloc[0]
    area_sqft = projected.area
    perim_ft  = projected.length
    area_sqkm  = area_sqft / SQFT_PER_SQKM
    area_acres = area_sqft / SQFT_PER_ACRE
    perim_km   = perim_ft / FT_PER_KM
    radius_m   = WALKING_SPEED_M_PER_MIN * minutes
    euclidean_area_sqkm = math.pi * (radius_m / 1000.0) ** 2
    efficiency = (area_sqkm / euclidean_area_sqkm
                  if euclidean_area_sqkm > 0 else float('nan'))
    # Compactness: 4*pi*area / perimeter^2 -> 1.0 for a circle, lower for irregular.
    compactness = ((4 * math.pi * area_sqft) / (perim_ft ** 2)
                   if perim_ft > 0 else float('nan'))
    metric_rows.append({
        "minutes":                  minutes,
        "walkshed_area_sqkm":       round(area_sqkm, 3),
        "walkshed_area_acres":      round(area_acres, 1),
        "euclidean_area_sqkm":      round(euclidean_area_sqkm, 3),
        "network_efficiency_ratio": round(efficiency, 3),
        "perimeter_km":             round(perim_km, 2),
        "compactness_ratio":        round(compactness, 3),
    })

walkshed_metrics_df = pd.DataFrame(metric_rows)
print("Walkshed metrics by threshold:")
display(walkshed_metrics_df)
out_csv = os.path.join(OUTPUT_FOLDER, "walksheds", "walkshed_metrics.csv")
walkshed_metrics_df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

In [ ]:
# Barrier detection — convex hull area vs. actual polygon area.
if 15 in walkshed_polygons:
    walkshed_15 = walkshed_polygons[15]
    convex_hull = walkshed_15.convex_hull
    proj_walkshed = gpd.GeoSeries([walkshed_15], crs="EPSG:4326").to_crs(epsg=2263).iloc[0]
    proj_hull     = gpd.GeoSeries([convex_hull],   crs="EPSG:4326").to_crs(epsg=2263).iloc[0]
    walkshed_area = proj_walkshed.area
    hull_area     = proj_hull.area
    fragmentation_index = ((hull_area - walkshed_area) / hull_area
                           if hull_area > 0 else float('nan'))
    print(f"15-minute walkshed area (sq ft): {walkshed_area:,.0f}")
    print(f"Convex hull area      (sq ft): {hull_area:,.0f}")
    print(f"Fragmentation index           : {fragmentation_index:.3f}")
    print()
    if fragmentation_index < 0.1:
        verdict = "Low fragmentation — walkshed is roughly convex, few barriers."
    elif fragmentation_index < 0.25:
        verdict = ("Moderate fragmentation — likely some barriers, but the "
                   "walkshed is reasonably compact.")
    else:
        verdict = ("High fragmentation — the walkshed is being cut by a physical "
                   "barrier. If you are near a waterfront, highway, or rail yard "
                   "this is expected. In a dense urban grid it may indicate "
                   "missing pedestrian infrastructure in the underlying OSM data.")
    print(verdict)
else:
    print("No 15-min walkshed available for barrier detection.")

In [ ]:
# Folium map: nested walksheds + Euclidean reference circles + convex hull outline.
center = [origin_lat, origin_lng]
walkshed_map = folium.Map(location=center, zoom_start=14, tiles="cartodbpositron")

# Nested walkshed polygons, light-to-dark.
ordered_minutes = sorted(walkshed_polygons.keys(), reverse=True)
for index, minutes in enumerate(ordered_minutes):
    fraction = (index + 1) / len(ordered_minutes)
    fill = "#%02x%02x%02x" % tuple(
        int(255 - (255 - channel) * fraction) for channel in (60, 78, 214))
    folium.GeoJson(walkshed_polygons[minutes].__geo_interface__,
                   style_function=lambda feature, c=fill: {"fillColor": c,
                                                            "color": c,
                                                            "weight": 1.2,
                                                            "fillOpacity": 0.32},
                   tooltip=f"{minutes}-min walkshed").add_to(walkshed_map)

# Euclidean reference circles (dashed).
for minutes in walkshed_polygons:
    radius_m = WALKING_SPEED_M_PER_MIN * minutes
    folium.Circle(location=center, radius=radius_m, fill=False,
                  color="#999999", weight=1.5, dash_array="5, 5",
                  tooltip=f"Euclidean {minutes} min").add_to(walkshed_map)

# Convex hull of the 15-min walkshed (dotted).
if 15 in walkshed_polygons:
    folium.GeoJson(walkshed_polygons[15].convex_hull.__geo_interface__,
                   style_function=lambda feature: {"fillOpacity": 0,
                                                    "color": "#E67E22",
                                                    "weight": 2,
                                                    "dashArray": "2, 6"},
                   tooltip="15-min convex hull").add_to(walkshed_map)

folium.Marker(location=center, popup=ORIGIN_ADDRESS,
              icon=folium.Icon(color="darkblue", icon="home")).add_to(walkshed_map)

map_path = os.path.join(OUTPUT_FOLDER, "maps", "walkshed_map.html")
walkshed_map.save(map_path)
print(f"Saved: {map_path}")
display(walkshed_map)

In [ ]:
# Network efficiency vs. time threshold.
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(walkshed_metrics_df["minutes"],
        walkshed_metrics_df["network_efficiency_ratio"],
        marker="o", linewidth=2, color="#3C4ED6")
ax.axhline(1.0, color="#999", linestyle="--", linewidth=1)
ax.set_xlabel("Walking time threshold (minutes)")
ax.set_ylabel("Network efficiency ratio (walkshed area / Euclidean area)")
ax.set_title(f"Network efficiency vs. time threshold — {ORIGIN_ADDRESS}")
ax.set_ylim(0, 1.05)
ax.annotate("1.0 = perfect omnidirectional access\n(no streets, no barriers)",
            xy=(walkshed_metrics_df["minutes"].iloc[-1], 1.0),
            xytext=(walkshed_metrics_df["minutes"].iloc[-1]-8, 0.92),
            fontsize=9, color="#666")
ax.annotate("Lower values =\nmore constrained by network",
            xy=(walkshed_metrics_df["minutes"].iloc[0], 0.4),
            xytext=(walkshed_metrics_df["minutes"].iloc[0]+1, 0.3),
            fontsize=9, color="#666")
plt.tight_layout()
chart_path = os.path.join(OUTPUT_FOLDER, "walksheds", "efficiency_chart.png")
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {chart_path}")

**What this analysis cannot tell you.**

- **Whether sidewalks are safe, well-lit, or maintained.** A 15-minute walkshed on paper is not a 15-minute walkshed for a woman walking home at midnight, a parent pushing a stroller over broken pavement, or anyone who has reason to avoid a particular block.
- **Whether pedestrian signals are timed appropriately.** Two intersections can be the same network distance but very different in actual walking time depending on signal cycles, mid-block crossings, and beg-button policies.
- **Whether the walkshed is passable for wheelchair users.** OSM's `wheelchair=yes/no/limited` tagging is incomplete in NYC, especially in outer boroughs. A walkshed that includes a flight of subway stairs is no walkshed for many residents.
- **Whether perceived safety affects actual route choice.** Routes that look short on the map are routinely avoided by real residents. The shortest path is not the same as the chosen path.

---

## Module 4 — Accessibility Scoring Overview and Application

This module introduces **accessibility scoring** as a method, surveys the major frameworks, and applies one — the cumulative-opportunity measure — to the origin's 15-minute walkshed.

**Module 4 depends on:** Module 3 (uses the 15-min walkshed polygon). Will re-fetch if necessary.

### What is accessibility scoring?

**Accessibility** is the ease with which people can reach destinations they need or want. It is a function of two things:

1. **Land use** — what destinations exist, where, in what quantity, and of what quality
2. **Transportation** — how easy it is to get to those destinations, given the modes available

Accessibility is *not* the same as:
- **Mobility** (how far or fast you can travel) — you can have high mobility and low accessibility if there is nothing worth reaching at the end of your trip
- **Proximity** (how close things are in Euclidean distance) — proximity is a one-variable abstraction; accessibility incorporates the network and the destinations

### The four major frameworks

**1. Cumulative opportunity measure.** Count the number of destinations of a given type reachable within a time threshold. *"Jobs reachable within 45 minutes by transit."* Simple, intuitive, and binary — a job 5 minutes away and a job 44 minutes away count equally; the 46-minute job does not count at all. This is what most public-facing accessibility analyses use, and it is what we use in this module.

**2. Gravity-based measure.** Weight each destination by its distance using a decay function (typically exponential or inverse-power). Closer destinations count more. More theoretically grounded than cumulative opportunity; harder to explain to non-technical audiences.

**3. Utility-based measure.** Use a discrete-choice model to simulate which destination a "rational" traveler would choose given travel time, destination size, and competition from other travelers. Most theoretically rigorous of the four; most data-intensive; requires survey or revealed-preference data to calibrate.

**4. People-based (activity) measure.** Follow individuals through their daily schedule and measure the destinations they can realistically reach given time constraints, trip chaining, and household responsibilities. Captures the difference between a single person's access and a parent who must drop off children before commuting. The most realistic of the four; almost never feasible with public data alone.

**Most open-data accessibility analyses use cumulative-opportunity measures because the data requirements are manageable. This notebook does the same.** Keep the other frameworks in mind as a reminder of what we are simplifying.

### Key references worth knowing by name

- **Jarrett Walker, *Human Transit* (2011)** — the working planner's primer on the relationship between network structure and access. Walker's blog (humantransit.org) is also essential.
- **The Accessibility Observatory** at the University of Minnesota — publishes annual transit-accessibility benchmarks for US metropolitan areas. https://access.umn.edu/
- **RPA's Fourth Regional Plan** — region-specific accessibility gap analysis for the New York / NJ / CT metropolitan area
- **Ahmed El-Geneidy and David Levinson, "Measuring Urban Accessibility"** — methodological primer that lays out the four frameworks above in more depth

In [ ]:
# Module 4 — fetch destinations, count within 15-min walkshed, score.
import os
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, shape
from tqdm.auto import tqdm

# Recover (or re-fetch) the 15-minute walkshed polygon.
try:
    walkshed_15 = walkshed_polygons.get(15)
except NameError:
    walkshed_15 = None

if walkshed_15 is None:
    print("Re-fetching the 15-min walkshed because Module 3 was not run in this session...")
    if ORS_API_KEY and ORIGIN_LATLNG is not None:
        walkshed_15 = make_ors_isochrone(
            (ORIGIN_LATLNG[1], ORIGIN_LATLNG[0]), "foot-walking", 15, ORS_API_KEY)
if walkshed_15 is None:
    print("Cannot proceed without a 15-min walkshed. Re-run Modules 0 and 3.")
    raise SystemExit

walkshed_gdf = gpd.GeoDataFrame(geometry=[walkshed_15], crs="EPSG:4326")

def _fetch_socrata(endpoint, params=None, max_rows=20000,
                  domain="data.cityofnewyork.us"):
    """Minimal paginated Socrata fetch for destination datasets."""
    base_url = f"https://{domain}/resource/{endpoint}"
    params = dict(params or {})
    rows = []
    page_size = 5000
    for offset in range(0, max_rows, page_size):
        params["$limit"]  = min(page_size, max_rows - offset)
        params["$offset"] = offset
        try:
            response = requests.get(base_url, params=params, timeout=60)
            response.raise_for_status()
            batch = response.json()
        except Exception as fetch_error:
            print(f"  Fetch failed at offset {offset}: {fetch_error}")
            break
        if not batch:
            break
        rows.extend(batch)
        if len(batch) < page_size:
            break
    return pd.DataFrame(rows)

print("Fetching destination datasets from NYC Open Data and NY State Open Data...")

In [ ]:
# Destination 1 — NYC Parks (polygons; we use centroids for counting).
parks_df = _fetch_socrata("enfh-gkve.json", max_rows=5000)
if "multipolygon" in parks_df.columns:
    parks_df["geometry"] = parks_df["multipolygon"].apply(
        lambda raw_geom: shape(raw_geom) if isinstance(raw_geom, dict) else None)
    parks_gdf = gpd.GeoDataFrame(
        parks_df.dropna(subset=["geometry"]),
        geometry="geometry", crs="EPSG:4326")
    # Count by centroid-in-polygon for consistency with point datasets.
    parks_gdf["geometry"] = parks_gdf.geometry.centroid
else:
    parks_gdf = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
print(f"Parks (points from centroid): {len(parks_gdf)}")

# Destination 2 — Public schools (Facilities DB filter — current schools dataset
# the spec referenced (p6h4-mpfy) is 404; the Facilities DB is the cleanest path).
schools_df = _fetch_socrata(
    "ji82-xba5.json",
    params={"$select": "facname,address,latitude,longitude",
            "$where":  "facsubgrp='PUBLIC K-12 SCHOOLS'"},
    max_rows=2000)
def _to_geo(point_df):
    valid = point_df.copy()
    valid["latitude"]  = pd.to_numeric(valid["latitude"],  errors="coerce")
    valid["longitude"] = pd.to_numeric(valid["longitude"], errors="coerce")
    valid = valid.dropna(subset=["latitude", "longitude"])
    valid["geometry"] = [Point(lng, lat)
                         for lng, lat in zip(valid["longitude"], valid["latitude"])]
    return gpd.GeoDataFrame(valid, geometry="geometry", crs="EPSG:4326")
schools_gdf = _to_geo(schools_df)
print(f"Public schools: {len(schools_gdf)}")

# Destination 3 — Public Libraries (Facilities DB filter; the spec's p4pf-fyc4 is empty).
libraries_df = _fetch_socrata(
    "ji82-xba5.json",
    params={"$select": "facname,address,latitude,longitude",
            "$where":  "facsubgrp='PUBLIC LIBRARIES'"},
    max_rows=500)
libraries_gdf = _to_geo(libraries_df)
print(f"Public libraries: {len(libraries_gdf)}")

# Destination 4 — Health Facilities (Facilities DB filter; spec's b8vs-jz6q is 404).
health_df = _fetch_socrata(
    "ji82-xba5.json",
    params={"$select": "facname,address,latitude,longitude,facsubgrp",
            "$where":  "facsubgrp IN ('HOSPITALS AND CLINICS','MENTAL HEALTH',"
                       "'OTHER HEALTH CARE')"},
    max_rows=5000)
health_gdf = _to_geo(health_df)
print(f"Health facilities: {len(health_gdf)}")

# Destination 5 — Grocery / retail via PLUTO landuse + bldgclass.
pluto_df = _fetch_socrata(
    "64uk-42ks.json",
    params={"$select": "bbl,address,landuse,bldgclass,latitude,longitude",
            "$where":  "landuse='05' AND (starts_with(bldgclass,'K') OR "
                       "starts_with(bldgclass,'S')) AND latitude IS NOT NULL"},
    max_rows=20000)
retail_gdf = _to_geo(pluto_df)
print(f"Retail / grocery lots: {len(retail_gdf)}")

# Destination 6 — Subway entrances from NY State Open Data (the spec's drh3-e2fd
# is empty; the MTA migrated their data to data.ny.gov).
subway_df = _fetch_socrata(
    "i9wp-a4ja.json",
    params={"$select": "station_id,stop_name,entrance_latitude,entrance_longitude"},
    max_rows=5000,
    domain="data.ny.gov")
if not subway_df.empty:
    subway_df["latitude"]  = pd.to_numeric(subway_df["entrance_latitude"],
                                           errors="coerce")
    subway_df["longitude"] = pd.to_numeric(subway_df["entrance_longitude"],
                                           errors="coerce")
subway_gdf = _to_geo(subway_df) if not subway_df.empty else gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
print(f"Subway entrances: {len(subway_gdf)}")

In [ ]:
# Count each destination type within the 15-min walkshed and tabulate.
# Citywide median densities below are rough planning-grade references (per sqkm),
# computed from total counts above divided by ~778 sqkm of NYC land area. They
# are not authoritative; treat them as order-of-magnitude benchmarks only.
NYC_LAND_AREA_SQKM = 778.0
SQFT_PER_SQKM = 10763910.41671

destinations = {
    "parks":            parks_gdf,
    "public_schools":   schools_gdf,
    "libraries":        libraries_gdf,
    "health_facilities":health_gdf,
    "retail_grocery":   retail_gdf,
    "subway_entrances": subway_gdf,
}

# Walkshed area in sqkm for density normalization.
walkshed_area_sqkm = (gpd.GeoSeries([walkshed_15], crs="EPSG:4326")
                     .to_crs(epsg=2263).area.iloc[0]) / SQFT_PER_SQKM

count_rows = []
for label, point_gdf in destinations.items():
    if point_gdf.empty:
        count_rows.append({"destination_type": label,
                           "count_within_walkshed": 0,
                           "density_per_sqkm": 0.0,
                           "nyc_median_density": round(len(point_gdf) / NYC_LAND_AREA_SQKM, 3),
                           "above_median": False})
        continue
    inside = gpd.sjoin(point_gdf, walkshed_gdf, how="inner", predicate="within")
    inside_count = len(inside)
    citywide_density = len(point_gdf) / NYC_LAND_AREA_SQKM
    local_density = (inside_count / walkshed_area_sqkm
                     if walkshed_area_sqkm > 0 else 0.0)
    count_rows.append({
        "destination_type":      label,
        "count_within_walkshed": int(inside_count),
        "density_per_sqkm":      round(local_density, 2),
        "nyc_median_density":    round(citywide_density, 3),
        "above_median":          local_density > citywide_density,
    })
access_score_df = pd.DataFrame(count_rows)
print(f"15-min walkshed area: {walkshed_area_sqkm:.3f} sq km\n")
print("Cumulative opportunity counts (15-min walkshed):")
display(access_score_df.style.background_gradient(
    subset=["count_within_walkshed"], cmap="Blues"))

out_csv = os.path.join(OUTPUT_FOLDER, "accessibility", "access_score.csv")
access_score_df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

In [ ]:
# Folium map: walkshed + each destination type as a toggleable layer.
DESTINATION_COLORS = {
    "parks":             "#117864",
    "public_schools":    "#3C4ED6",
    "libraries":         "#7B241C",
    "health_facilities": "#C0392B",
    "retail_grocery":    "#E67E22",
    "subway_entrances":  "#1B1B33",
}

access_map = folium.Map(location=[origin_lat, origin_lng],
                        zoom_start=14, tiles="cartodbpositron")
folium.GeoJson(walkshed_15.__geo_interface__,
               style_function=lambda feature: {"fillColor": "#3C4ED6",
                                                "color":     "#3C4ED6",
                                                "weight":    1.5,
                                                "fillOpacity": 0.15},
               tooltip="15-minute walkshed").add_to(access_map)

for label, point_gdf in destinations.items():
    if point_gdf.empty:
        continue
    inside = gpd.sjoin(point_gdf, walkshed_gdf, how="inner", predicate="within")
    layer = folium.FeatureGroup(name=f"{label} ({len(inside)} in walkshed)",
                                show=True)
    for _, row in inside.iterrows():
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=4, color=DESTINATION_COLORS[label],
            fill=True, fill_opacity=0.85,
            tooltip=row.get("facname", row.get("name", label))
        ).add_to(layer)
    layer.add_to(access_map)

folium.Marker(location=[origin_lat, origin_lng],
              popup=ORIGIN_ADDRESS,
              icon=folium.Icon(color="darkblue", icon="home")).add_to(access_map)
folium.LayerControl(collapsed=False).add_to(access_map)

map_path = os.path.join(OUTPUT_FOLDER, "accessibility", "access_score_map.html")
access_map.save(map_path)
print(f"Saved: {map_path}")
display(access_map)

**What this analysis cannot tell you.**

- **Whether destinations are affordable, open, or welcoming to all residents.** A grocery store that primarily stocks expensive prepared foods, a clinic that does not accept Medicaid, a park whose benches have been removed to discourage loitering — these all count as one destination here.
- **Whether destinations are accessible to people with disabilities.** An ADA-non-compliant subway entrance counts the same as an elevator station.
- **Whether destination quality varies systematically by neighborhood.** A 200-acre flagship park and a 0.1-acre vest-pocket tot lot both contribute one to the park count. A two-room storefront branch library and a flagship NYPL branch both contribute one. The map says these destinations exist; it does not say what kind of resource they are.
- **Whether people actually use these destinations.** Potential access is not realized access. Module 5 makes the gap between the two more visible by comparing across neighborhoods.

---

## Module 5 — Neighborhood Comparison

This is the **equity module**. We compute 15-minute walking isochrones and Google-transit travel times from each of the five `COMPARISON_ADDRESSES`, and overlay them at a common spatial scale so the differences are unmistakable.

**Read this before the maps render.** This is not a ranking of neighborhoods on a goodness scale. It is a **visualization of structural disinvestment in transit infrastructure** in a city where transit was the foundational infrastructure of urban form. The locations are chosen deliberately:

- **Times Square (Midtown Manhattan)** — the most transit-rich location in the western hemisphere. Several subway lines converge here; PATH connects to New Jersey; intercity rail is two blocks away.
- **165 Rockaway Ave (Brownsville, Brooklyn)** — one of the most transit-poor neighborhoods in NYC relative to population density and need. The 2017 NYC Comptroller's report *The Other Transit Crisis* and the 2019 Regional Plan Association report *Five Borough Bikeway* both flag Brownsville and adjacent neighborhoods as critical access gaps.
- **149th St / Grand Concourse (South Bronx)** — frequent service on the 2, 4, 5 corridor but very poor crosstown connections; **transit-rich on one axis, transit-poor on the orthogonal axis**.
- **Jamaica Center (Queens)** — a major transit hub for southeastern Queens (E, J, Z, LIRR, dozens of bus routes), but surrounded by transit-poor residential blocks. **Hub-and-spoke geography concentrated at one node.**
- **St George Ferry Terminal (Staten Island)** — Staten Island is the only NYC borough without a subway connection. The SIR (Staten Island Railway) and the Staten Island Ferry are the two backbones.

The literature on transit equity in NYC is large and worth knowing by name even if it is not linked here. Key sources include the **NYC Comptroller's office** transit reports, **RPA's** Fourth Regional Plan and access gap series, **TransitCenter's** annual rider surveys, **the Pratt Center for Community Development's** Tale of Two Cities transit work, and the **MTA's own** Inspector General reports on subway accessibility. Hold these in your head while you read the maps below.

In [ ]:
# Module 5 — fetch ORS walking + cycling and Google transit for all comparison
# addresses, plus time-of-day for the single origin.
import os
import time
import pandas as pd
import geopandas as gpd
import folium
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, shape
from tqdm.auto import tqdm

if not ORS_API_KEY:
    print("ORS_API_KEY missing — Module 5 cannot fetch walking isochrones.")
    raise SystemExit
if not GOOGLE_API_KEY:
    print("WARNING: GOOGLE_API_KEY missing — time-of-day comparison will be skipped.")

# Step 1: geocode every comparison address.
print("Geocoding comparison addresses...")
comparison_origins = {}
for address in COMPARISON_ADDRESSES:
    coords = geocode_address(address, GOOGLE_API_KEY)
    if coords is not None:
        comparison_origins[address] = coords
        print(f"  OK  {address} -> {coords}")
    else:
        print(f"  --  {address}: could not geocode")

# Step 2: fetch a 15-min ORS walking isochrone + cycling isochrone for each.
print("\nFetching ORS walking + cycling isochrones...")
comparison_walksheds = {}
comparison_bikesheds = {}
for address, (lat, lng) in tqdm(comparison_origins.items(), desc="locations"):
    walk_polygon = make_ors_isochrone((lng, lat), "foot-walking",
                                      15, ORS_API_KEY)
    bike_polygon = make_ors_isochrone((lng, lat), "cycling-regular",
                                      15, ORS_API_KEY)
    if walk_polygon is not None: comparison_walksheds[address] = walk_polygon
    if bike_polygon is not None: comparison_bikesheds[address] = bike_polygon
    time.sleep(1.0)
print(f"Walksheds fetched: {len(comparison_walksheds)}")

In [ ]:
# Step 3: count destinations within each walkshed (reuses Module 4 destinations).
# This requires Module 4 has been run. If not, we re-fetch silently.
try:
    destinations
except NameError:
    print("Re-running Module 4 destination fetches because they are not in memory...")
    # Inline minimal re-fetch — copy of Module 4 logic, condensed.
    parks_df = _fetch_socrata("enfh-gkve.json", max_rows=5000)
    parks_df["geometry"] = parks_df["multipolygon"].apply(
        lambda raw_geom: shape(raw_geom) if isinstance(raw_geom, dict) else None)
    parks_gdf = gpd.GeoDataFrame(parks_df.dropna(subset=["geometry"]),
                                 geometry="geometry", crs="EPSG:4326")
    parks_gdf["geometry"] = parks_gdf.geometry.centroid
    def _to_geo(df):
        v = df.copy()
        v["latitude"]  = pd.to_numeric(v["latitude"],  errors="coerce")
        v["longitude"] = pd.to_numeric(v["longitude"], errors="coerce")
        v = v.dropna(subset=["latitude","longitude"])
        v["geometry"] = [Point(lng,lat) for lng,lat in zip(v["longitude"],v["latitude"])]
        return gpd.GeoDataFrame(v, geometry="geometry", crs="EPSG:4326")
    schools_gdf = _to_geo(_fetch_socrata("ji82-xba5.json",
        params={"$select":"facname,latitude,longitude",
                "$where":"facsubgrp='PUBLIC K-12 SCHOOLS'"}, max_rows=2000))
    libraries_gdf = _to_geo(_fetch_socrata("ji82-xba5.json",
        params={"$select":"facname,latitude,longitude",
                "$where":"facsubgrp='PUBLIC LIBRARIES'"}, max_rows=500))
    health_gdf = _to_geo(_fetch_socrata("ji82-xba5.json",
        params={"$select":"facname,latitude,longitude",
                "$where":"facsubgrp IN ('HOSPITALS AND CLINICS','MENTAL HEALTH','OTHER HEALTH CARE')"},
        max_rows=5000))
    retail_gdf = _to_geo(_fetch_socrata("64uk-42ks.json",
        params={"$select":"bbl,landuse,bldgclass,latitude,longitude",
                "$where":"landuse='05' AND (starts_with(bldgclass,'K') OR starts_with(bldgclass,'S')) AND latitude IS NOT NULL"},
        max_rows=20000))
    subway_raw = _fetch_socrata("i9wp-a4ja.json",
        params={"$select":"station_id,stop_name,entrance_latitude,entrance_longitude"},
        max_rows=5000, domain="data.ny.gov")
    subway_raw["latitude"]  = pd.to_numeric(subway_raw["entrance_latitude"],errors="coerce")
    subway_raw["longitude"] = pd.to_numeric(subway_raw["entrance_longitude"],errors="coerce")
    subway_gdf = _to_geo(subway_raw)
    destinations = {"parks":parks_gdf, "public_schools":schools_gdf,
                    "libraries":libraries_gdf, "health_facilities":health_gdf,
                    "retail_grocery":retail_gdf, "subway_entrances":subway_gdf}

print("Counting destinations within each comparison walkshed...")
SQFT_PER_SQKM = 10763910.41671
comparison_rows = []
for address, walk_polygon in tqdm(comparison_walksheds.items(), desc="counts"):
    walk_gdf = gpd.GeoDataFrame(geometry=[walk_polygon], crs="EPSG:4326")
    area_sqkm = gpd.GeoSeries([walk_polygon], crs="EPSG:4326").to_crs(epsg=2263).area.iloc[0] / SQFT_PER_SQKM
    counts = {"address": address, "walkshed_area_sqkm": round(area_sqkm, 3)}
    for label, point_gdf in destinations.items():
        if point_gdf.empty:
            counts[label] = 0
            continue
        inside = gpd.sjoin(point_gdf, walk_gdf, how="inner", predicate="within")
        counts[label] = int(len(inside))
    comparison_rows.append(counts)

comparison_df = pd.DataFrame(comparison_rows)
print("\n15-minute walkshed comparison:")
display(comparison_df)

In [ ]:
# Step 4: time-of-day Google transit. The user-selected scope is "single origin,
# multiple times" — we vary the time of day for the configured ORIGIN_ADDRESS to
# a fixed set of comparison destinations (the five comparison locations).
import datetime
if not GOOGLE_API_KEY:
    print("Skipping time-of-day comparison — no Google key.")
    time_of_day_df = pd.DataFrame()
else:
    # Build three future departure timestamps (next weekday 8am, 12pm, 9pm).
    def _next_weekday(hour):
        """Return Unix timestamp for the next weekday at the given local hour."""
        now = datetime.datetime.now()
        target = now.replace(hour=hour, minute=0, second=0, microsecond=0)
        if target <= now:
            target += datetime.timedelta(days=1)
        while target.weekday() >= 5:  # 5,6 = Sat, Sun
            target += datetime.timedelta(days=1)
        return int(target.timestamp())

    departure_times = {
        "peak_am":  _next_weekday(8),
        "midday":   _next_weekday(12),
        "evening":  _next_weekday(21),
    }

    print("Calling Google Distance Matrix (transit) for origin -> each comparison location, "
          "at 3 times of day...")
    destination_list = list(comparison_origins.items())
    destination_coords = [coords for _, coords in destination_list]
    if not destination_list:
        time_of_day_df = pd.DataFrame()
    else:
        per_time_results = {}
        for label, ts in departure_times.items():
            result_df = google_distance_matrix(
                origins=[ORIGIN_LATLNG],
                destinations=destination_coords,
                mode="transit",
                api_key=GOOGLE_API_KEY,
                departure_time=ts)
            per_time_results[label] = result_df
            time.sleep(1.0)

        rows = []
        for index, (address, _) in enumerate(destination_list):
            row = {"origin": ORIGIN_ADDRESS, "destination": address}
            for label, df in per_time_results.items():
                if df.empty:
                    row[f"{label}_min"] = None
                    continue
                match = df.query(f"destination_index == {index}")
                row[f"{label}_min"] = (round(match["travel_minutes"].iloc[0], 1)
                                       if not match.empty else None)
            peak = row.get("peak_am_min")
            ev   = row.get("evening_min")
            row["peak_to_evening_delta_min"] = (round(ev - peak, 1)
                                                if peak is not None and ev is not None
                                                else None)
            rows.append(row)
        time_of_day_df = pd.DataFrame(rows)
        print("\nTransit travel time from origin to each comparison location:")
        display(time_of_day_df)

        out_csv = os.path.join(OUTPUT_FOLDER, "timeofday",
                               "time_of_day_comparison.csv")
        time_of_day_df.to_csv(out_csv, index=False)
        print(f"\nSaved: {out_csv}")

In [ ]:
# Visualization 1: small multiples of 15-min walking isochrones at common scale.
import matplotlib.patches as mpatches
import matplotlib.transforms as transforms

n_locations = len(comparison_walksheds)
if n_locations == 0:
    print("No comparison walksheds to plot.")
else:
    rows = 2 if n_locations > 3 else 1
    cols = math.ceil(n_locations / rows)
    fig, axes = plt.subplots(rows, cols, figsize=(4.5*cols, 4.5*rows))
    if n_locations == 1:
        axes = np.array([axes])
    axes_flat = axes.flatten()

    # Common scale: largest bounding box across all polygons, padded.
    all_minx = []; all_miny = []; all_maxx = []; all_maxy = []
    for polygon in comparison_walksheds.values():
        minx, miny, maxx, maxy = polygon.bounds
        all_minx.append(minx); all_miny.append(miny)
        all_maxx.append(maxx); all_maxy.append(maxy)
    pad = 0.005
    xlim = (min(all_minx) - pad, max(all_maxx) + pad)
    ylim = (min(all_miny) - pad, max(all_maxy) + pad)

    for axis, (address, polygon) in zip(axes_flat, comparison_walksheds.items()):
        poly_gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
        poly_gdf.plot(ax=axis, facecolor="#3C4ED6", edgecolor="#1B1B33",
                      alpha=0.45, linewidth=1.2)
        origin_xy = comparison_origins[address]
        axis.plot(origin_xy[1], origin_xy[0], "o",
                  color="#7B241C", markersize=8, zorder=5)
        axis.set_xlim(xlim); axis.set_ylim(ylim)
        axis.set_aspect("equal")
        # Strip the address down to its first segment for the title.
        short_title = address.split(",")[0]
        axis.set_title(short_title, fontsize=10)
        axis.set_xticks([]); axis.set_yticks([])
    for axis in axes_flat[n_locations:]:
        axis.axis("off")
    fig.suptitle("15-minute walking isochrones — same scale, same time, very different cities",
                 fontsize=12, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    path = os.path.join(OUTPUT_FOLDER, "comparison", "small_multiples.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path}")

In [ ]:
# Visualization 2: walkshed area bar chart, sorted, with citywide-median reference.
sorted_df = comparison_df.sort_values("walkshed_area_sqkm", ascending=True).copy()
median_area = sorted_df["walkshed_area_sqkm"].median()

fig, ax = plt.subplots(figsize=(10, 5))
short_labels = [a.split(",")[0] for a in sorted_df["address"]]
bars = ax.barh(short_labels, sorted_df["walkshed_area_sqkm"], color="#3C4ED6")
ax.axvline(median_area, color="#C0392B", linestyle="--", linewidth=1.5,
           label=f"5-location median = {median_area:.2f} sq km")
ax.set_xlabel("15-minute walking isochrone area (sq km)")
ax.set_title("15-minute walkshed area by location")
ax.legend()
plt.tight_layout()
bar_path = os.path.join(OUTPUT_FOLDER, "comparison", "walkshed_areas.png")
plt.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {bar_path}")

In [ ]:
# Visualization 3: destination access heatmap.
heatmap_cols = ["parks","public_schools","libraries","health_facilities",
                "retail_grocery","subway_entrances"]
heatmap_data = comparison_df.set_index("address")[heatmap_cols].copy()
heatmap_data.index = [a.split(",")[0] for a in heatmap_data.index]

fig, ax = plt.subplots(figsize=(10, 4 + 0.25*len(heatmap_data)))
sns.heatmap(heatmap_data, annot=True, fmt="d", cmap="Blues",
            cbar_kws={"label": "Destinations within 15-min walkshed"},
            ax=ax, linewidths=0.5, linecolor="white")
ax.set_xlabel("Destination type")
ax.set_ylabel("Origin")
ax.set_title("Destination access in 15-minute walksheds (cumulative opportunity)")
plt.tight_layout()
heat_path = os.path.join(OUTPUT_FOLDER, "comparison", "access_heatmap.png")
plt.savefig(heat_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {heat_path}")

In [ ]:
# Visualization 4: time-of-day variation chart (single origin -> 5 destinations).
if 'time_of_day_df' in dir() and not time_of_day_df.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    short_labels = [d.split(",")[0] for d in time_of_day_df["destination"]]
    x_positions = np.arange(len(short_labels))
    width = 0.27
    ax.bar(x_positions - width, time_of_day_df["peak_am_min"], width,
           label="Peak AM (8am)", color="#3C4ED6")
    ax.bar(x_positions,         time_of_day_df["midday_min"], width,
           label="Midday (12pm)", color="#16A085")
    ax.bar(x_positions + width, time_of_day_df["evening_min"], width,
           label="Evening (9pm)", color="#E67E22")
    ax.set_xticks(x_positions); ax.set_xticklabels(short_labels, rotation=20, ha="right")
    ax.set_ylabel("Transit travel time (minutes)")
    ax.set_title(f"Transit time from {ORIGIN_ADDRESS.split(',')[0]} to comparison locations\nat three departure times")
    ax.legend()
    plt.tight_layout()
    tod_path = os.path.join(OUTPUT_FOLDER, "timeofday", "time_of_day_chart.png")
    plt.savefig(tod_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {tod_path}")
else:
    print("No time-of-day data available — Google key missing or call failed.")

In [ ]:
# Visualization 5: interactive folium map with all five isochrones as toggles.
center_lat = np.mean([coords[0] for coords in comparison_origins.values()])
center_lng = np.mean([coords[1] for coords in comparison_origins.values()])
comp_map = folium.Map(location=[center_lat, center_lng],
                      zoom_start=11, tiles="cartodbpositron")

palette = ["#3C4ED6", "#C0392B", "#16A085", "#E67E22", "#7B241C", "#9B59B6"]
counts_by_address = comparison_df.set_index("address").to_dict("index")

for index, (address, polygon) in enumerate(comparison_walksheds.items()):
    color = palette[index % len(palette)]
    counts = counts_by_address.get(address, {})
    tooltip_lines = [f"<b>{address}</b>",
                     f"Walkshed area: {counts.get('walkshed_area_sqkm','?')} sq km"]
    for label in ["parks","public_schools","libraries",
                  "health_facilities","retail_grocery","subway_entrances"]:
        tooltip_lines.append(f"{label}: {counts.get(label, 0)}")
    tooltip_html = "<br>".join(tooltip_lines)
    group = folium.FeatureGroup(name=address.split(",")[0], show=True)
    folium.GeoJson(polygon.__geo_interface__,
                   style_function=lambda feature, c=color: {"fillColor": c,
                                                             "color": c,
                                                             "weight": 1.5,
                                                             "fillOpacity": 0.35},
                   tooltip=folium.Tooltip(tooltip_html)).add_to(group)
    origin_xy = comparison_origins[address]
    folium.CircleMarker(location=[origin_xy[0], origin_xy[1]],
                        radius=5, color=color, fill=True,
                        fill_opacity=1.0).add_to(group)
    group.add_to(comp_map)

folium.LayerControl(collapsed=False).add_to(comp_map)
comp_map_path = os.path.join(OUTPUT_FOLDER, "maps", "comparison_map.html")
comp_map.save(comp_map_path)
print(f"Saved: {comp_map_path}")
display(comp_map)

**Look at the small multiples.** The isochrones are drawn at the same scale. **What do the size differences mean for daily life?**

- The Brownsville walkshed and the Midtown walkshed both represent 15 minutes of walking. **What is different about what those 15 minutes can reach?** Hold the destination heatmap next to the small multiples and you should be able to see the answer in two glances.

- **Time-of-day variation is largest in which location?** That tells you about transit frequency and who bears the burden of infrequent service. A 15-minute walk does not change between 8am and 9pm. A transit trip absolutely can.

- **If you were a parent in the lowest-access neighborhood**, which destination gap would matter most to you? Now ask yourself: when was the last time you saw a transit plan that started with that parent's question?

**What this analysis cannot tell you.**

- **It cannot tell you whether the access gap is structural or temporal.** A neighborhood with low access at 9pm but high access at 8am has a service-frequency problem; one with low access at all hours has an infrastructure problem. They require different policy responses.
- **It cannot tell you whether the destinations within reach are usable.** A library on the walkshed boundary that closes at 5pm has less practical reach than a library across town that is open until 8.
- **It cannot tell you what the residents would have measured if they had built this analysis themselves.** Co-production is not a label — it is what would make these maps actually useful.

---

## Module 6 — Student Exploration

You have seen isochrones, walksheds, accessibility scores, and a neighborhood comparison. Now **you** choose the origins, the mode, the threshold, the destinations, and the question.

The method is the same. The question is yours. Start by writing it down.

In [ ]:
# Module 6 — interactive exploration builder.
import os
import datetime
import json
import time
import folium
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Step 1 — research question.
question_input = widgets.Textarea(
    value="", placeholder=(
        "Example: How does transit access to hospitals differ between the South "
        "Bronx and Flushing? — or — What can you reach in 15 minutes from a NYCHA "
        "development in Brownsville vs. a market-rate building in Williamsburg? — "
        "or — How does evening transit service change the accessible job market "
        "for a night-shift worker in Jamaica?"),
    description="Your question:",
    layout=widgets.Layout(width="800px", height="80px"),
    style={"description_width": "initial"})

# Step 2 — up to four origins.
origin_inputs = [
    widgets.Text(value="", placeholder=f"Origin {i+1} address",
                 description=f"Origin {i+1}:",
                 layout=widgets.Layout(width="700px"),
                 style={"description_width": "initial"})
    for i in range(4)
]

# Step 3 — mode and threshold.
mode_picker = widgets.Dropdown(
    options=[("Walking","foot-walking"),
             ("Cycling","cycling-regular"),
             ("Driving","driving-car")],
    value="foot-walking", description="Mode:",
    style={"description_width": "initial"})
threshold_slider = widgets.IntSlider(
    value=15, min=5, max=45, step=5,
    description="Threshold (min):",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"))

# Step 4 — destination types.
destination_options = ["parks", "public_schools", "libraries",
                       "health_facilities", "retail_grocery", "subway_entrances"]
destination_picker = widgets.SelectMultiple(
    options=destination_options,
    value=("parks", "public_schools", "subway_entrances"),
    description="Destinations:",
    layout=widgets.Layout(width="500px"),
    style={"description_width": "initial"})

# Step 5 — cannot-tell-you reflection input.
limitations_input = widgets.Textarea(
    value="", placeholder="What does this analysis not tell you?",
    description="Limitations:",
    layout=widgets.Layout(width="800px", height="80px"),
    style={"description_width": "initial"})

run_button = widgets.Button(description="Run exploration",
                            button_style="primary", icon="play")
output_panel = widgets.Output()

def _make_exploration_card(payload):
    """Format the student's choices and findings as a plain-text card."""
    lines = ["=" * 76,
             f"EXPLORATION CARD",
             "=" * 76,
             f"Question        : {payload['question']}",
             f"Mode            : {payload['mode']}",
             f"Time threshold  : {payload['threshold_minutes']} minutes",
             f"Origins analyzed:"]
    for index, origin in enumerate(payload["origins"], 1):
        lines.append(f"  {index}. {origin}")
    lines.append(f"Destination types counted: {', '.join(payload['destinations'])}")
    lines.append("")
    lines.append("Key findings:")
    if payload["findings"].get("largest"):
        l = payload["findings"]["largest"]
        lines.append(f"  Largest walkshed : {l['origin']} "
                     f"({l['area_sqkm']} sq km)")
    if payload["findings"].get("smallest"):
        s = payload["findings"]["smallest"]
        lines.append(f"  Smallest walkshed: {s['origin']} "
                     f"({s['area_sqkm']} sq km)")
    if payload["findings"].get("biggest_gap"):
        g = payload["findings"]["biggest_gap"]
        lines.append(f"  Biggest access gap: '{g['destination']}' "
                     f"(max {g['max']} vs. min {g['min']})")
    if payload["findings"].get("largest_tod_delta"):
        t = payload["findings"]["largest_tod_delta"]
        lines.append(f"  Largest time-of-day delta: "
                     f"{t['origin']} ({t['delta']} min peak->evening)")
    lines.append("")
    lines.append("What this analysis cannot tell you:")
    lines.append(f"  {payload['limitations'] or '(left blank by the author)'}")
    lines.append("")
    lines.append(f"Generated: {payload['generated_at']}")
    lines.append("=" * 76)
    return "\n".join(lines)

def _on_run(_):
    with output_panel:
        output_panel.clear_output()
        if not ORS_API_KEY:
            print("Need ORS_API_KEY to run this module.")
            return
        # Collect non-empty origin addresses.
        origin_addresses = [box.value.strip() for box in origin_inputs
                            if box.value.strip()]
        if len(origin_addresses) == 0:
            print("Enter at least one origin address.")
            return
        if len(origin_addresses) > 4:
            origin_addresses = origin_addresses[:4]

        # Geocode all.
        origins_by_address = {}
        for address in origin_addresses:
            coords = geocode_address(address, GOOGLE_API_KEY)
            if coords is not None:
                origins_by_address[address] = coords
            else:
                print(f"Could not geocode '{address}' — skipping.")
        if not origins_by_address:
            print("No origins could be geocoded.")
            return

        chosen_mode = mode_picker.value
        chosen_threshold = int(threshold_slider.value)
        chosen_destinations = list(destination_picker.value)

        # Fetch isochrones.
        student_isochrones = {}
        for address, (lat, lng) in origins_by_address.items():
            polygon = make_ors_isochrone((lng, lat), chosen_mode,
                                         chosen_threshold, ORS_API_KEY)
            if polygon is not None:
                student_isochrones[address] = polygon
            time.sleep(1.0)
        if not student_isochrones:
            print("No isochrones could be fetched.")
            return

        # Count destinations within each.
        SQFT_PER_SQKM = 10763910.41671
        result_rows = []
        for address, polygon in student_isochrones.items():
            polygon_gdf = gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")
            area_sqkm = (gpd.GeoSeries([polygon], crs="EPSG:4326")
                         .to_crs(epsg=2263).area.iloc[0]) / SQFT_PER_SQKM
            row = {"origin": address, "area_sqkm": round(area_sqkm, 3)}
            for label in chosen_destinations:
                point_gdf = destinations.get(label)
                if point_gdf is None or point_gdf.empty:
                    row[label] = 0
                    continue
                inside = gpd.sjoin(point_gdf, polygon_gdf,
                                   how="inner", predicate="within")
                row[label] = int(len(inside))
            result_rows.append(row)
        result_df = pd.DataFrame(result_rows)
        display(result_df)

        # Time-of-day call from first origin to all others (if Google key).
        tod_df = pd.DataFrame()
        if GOOGLE_API_KEY and len(origins_by_address) >= 2:
            origins_list = list(origins_by_address.items())
            first_origin = origins_list[0][1]
            destination_coords = [c for _, c in origins_list[1:]]
            destination_addresses = [a for a, _ in origins_list[1:]]
            def _next_weekday(hour):
                now = datetime.datetime.now()
                t = now.replace(hour=hour, minute=0, second=0, microsecond=0)
                if t <= now: t += datetime.timedelta(days=1)
                while t.weekday() >= 5: t += datetime.timedelta(days=1)
                return int(t.timestamp())
            times = {"peak_am": _next_weekday(8),
                     "midday":  _next_weekday(12),
                     "evening": _next_weekday(21)}
            per_time = {}
            for label, ts in times.items():
                per_time[label] = google_distance_matrix(
                    origins=[first_origin],
                    destinations=destination_coords,
                    mode="transit", api_key=GOOGLE_API_KEY,
                    departure_time=ts)
                time.sleep(1.0)
            tod_rows = []
            for index, address in enumerate(destination_addresses):
                row = {"destination": address}
                for label, df in per_time.items():
                    if df.empty:
                        row[f"{label}_min"] = None; continue
                    match = df.query(f"destination_index == {index}")
                    row[f"{label}_min"] = (round(match["travel_minutes"].iloc[0], 1)
                                           if not match.empty else None)
                if row.get("peak_am_min") and row.get("evening_min"):
                    row["delta_min"] = round(row["evening_min"] - row["peak_am_min"], 1)
                tod_rows.append(row)
            tod_df = pd.DataFrame(tod_rows)
            if not tod_df.empty:
                print("\nTime-of-day transit from first origin:")
                display(tod_df)

        # Visualizations: small multiples + comparison map.
        n = len(student_isochrones)
        cols = min(n, 4); rows = math.ceil(n / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(4.5*cols, 4.5*rows))
        if n == 1:
            axes = np.array([axes])
        axes_flat = axes.flatten()
        # Common scale across all student origins.
        bx = [p.bounds for p in student_isochrones.values()]
        if bx:
            xlim = (min(b[0] for b in bx) - 0.005,
                    max(b[2] for b in bx) + 0.005)
            ylim = (min(b[1] for b in bx) - 0.005,
                    max(b[3] for b in bx) + 0.005)
            for axis, (address, polygon) in zip(axes_flat, student_isochrones.items()):
                gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326").plot(
                    ax=axis, facecolor="#3C4ED6", edgecolor="#1B1B33",
                    alpha=0.45, linewidth=1.2)
                coords = origins_by_address[address]
                axis.plot(coords[1], coords[0], "o", color="#7B241C",
                          markersize=8, zorder=5)
                axis.set_xlim(xlim); axis.set_ylim(ylim)
                axis.set_aspect("equal")
                axis.set_title(address.split(",")[0], fontsize=10)
                axis.set_xticks([]); axis.set_yticks([])
            for axis in axes_flat[n:]:
                axis.axis("off")
            plt.tight_layout()
            plt.show()

        # Findings for the card.
        sorted_areas = result_df.sort_values("area_sqkm", ascending=False)
        findings = {
            "largest":  {"origin": sorted_areas.iloc[0]["origin"],
                         "area_sqkm": sorted_areas.iloc[0]["area_sqkm"]},
            "smallest": {"origin": sorted_areas.iloc[-1]["origin"],
                         "area_sqkm": sorted_areas.iloc[-1]["area_sqkm"]},
        }
        if chosen_destinations and len(result_df) >= 2:
            spreads = {dest: int(result_df[dest].max() - result_df[dest].min())
                       for dest in chosen_destinations if dest in result_df.columns}
            if spreads:
                biggest = max(spreads, key=spreads.get)
                findings["biggest_gap"] = {
                    "destination": biggest,
                    "max": int(result_df[biggest].max()),
                    "min": int(result_df[biggest].min())}
        if not tod_df.empty and "delta_min" in tod_df.columns:
            valid_tod = tod_df.dropna(subset=["delta_min"])
            if not valid_tod.empty:
                row = valid_tod.iloc[valid_tod["delta_min"].abs().idxmax()]
                findings["largest_tod_delta"] = {
                    "origin": row["destination"],
                    "delta": row["delta_min"]}

        payload = {
            "question":          question_input.value.strip() or "(no question)",
            "origins":           list(origins_by_address.keys()),
            "mode":              chosen_mode,
            "threshold_minutes": chosen_threshold,
            "destinations":      chosen_destinations,
            "findings":          findings,
            "limitations":       limitations_input.value.strip(),
            "generated_at":      datetime.datetime.utcnow().isoformat(timespec="seconds")+"Z",
        }
        card_text = _make_exploration_card(payload)
        print(card_text)

        timestamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
        card_path = os.path.join(OUTPUT_FOLDER, "exports",
                                 f"exploration_card_{timestamp}.txt")
        with open(card_path, "w") as text_file:
            text_file.write(card_text)
        print(f"\nSaved exploration card: {card_path}")

        # Persist payload + isochrones to disk for Module 7.
        payload_path = os.path.join(OUTPUT_FOLDER, "exports",
                                    "exploration_payload.json")
        with open(payload_path, "w") as out_file:
            json.dump(payload, out_file, indent=2)
        features = []
        for address, polygon in student_isochrones.items():
            features.append({"type": "Feature",
                             "geometry": polygon.__geo_interface__,
                             "properties": {"origin": address,
                                            "mode": chosen_mode,
                                            "minutes": chosen_threshold}})
        with open(os.path.join(OUTPUT_FOLDER, "exports",
                               "exploration_isochrones.geojson"), "w") as out_file:
            json.dump({"type":"FeatureCollection", "features":features}, out_file)

run_button.on_click(_on_run)

display(widgets.VBox([
    widgets.HTML("<h4>Step 1 — Your question</h4>"),
    question_input,
    widgets.HTML("<h4>Step 2 — Origins (up to 4)</h4>"),
    *origin_inputs,
    widgets.HTML("<h4>Step 3 — Mode and threshold</h4>"),
    mode_picker, threshold_slider,
    widgets.HTML("<h4>Step 4 — Destination types</h4>"),
    destination_picker,
    widgets.HTML("<h4>Step 5 — What does this analysis not tell you?</h4>"),
    limitations_input,
    widgets.HTML("<h4>Step 6 — Run</h4>"),
    run_button, output_panel,
]))

---

## Module 7 — Export and Final Outputs

Bundle everything from earlier modules into a single set of exports and a single final map. Run this after you have completed at least Modules 0, 2, 3, and 4.

**Module 7 depends on:** outputs from Modules 2–6 written to `OUTPUT_FOLDER`.

In [ ]:
# Module 7 — bundle all isochrones into one GeoJSON, save final layered map.
import os
import json
import folium
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

# Step 1: assemble every isochrone we have on disk into one FeatureCollection.
SQFT_PER_SQKM = 10763910.41671
all_features = []

def _add_feature(polygon, origin_address, mode, source, minutes):
    """Append a single isochrone feature with computed metrics to all_features."""
    if polygon is None: return
    proj = gpd.GeoSeries([polygon], crs="EPSG:4326").to_crs(epsg=2263).iloc[0]
    area_sqkm = proj.area / SQFT_PER_SQKM
    perim_ft  = proj.length
    radius_m = 80.0 * int(minutes) if mode in ("foot-walking", "walking") else 250.0 * int(minutes)
    euclid = 3.141592653589793 * (radius_m / 1000.0) ** 2
    efficiency = (area_sqkm / euclid) if euclid > 0 else None
    compactness = ((4 * 3.141592653589793 * proj.area) / (perim_ft ** 2)
                   if perim_ft > 0 else None)
    all_features.append({
        "type": "Feature",
        "geometry": polygon.__geo_interface__,
        "properties": {
            "origin_address":           origin_address,
            "mode":                     mode,
            "api_source":               source,
            "threshold_minutes":        int(minutes),
            "walkshed_area_sqkm":       round(area_sqkm, 3),
            "network_efficiency_ratio": round(efficiency, 3) if efficiency else None,
            "compactness_ratio":        round(compactness, 3) if compactness else None,
        },
    })

# Module 2 ORS.
ors_path = os.path.join(OUTPUT_FOLDER, "isochrones", "ors_raw.geojson")
if os.path.exists(ors_path):
    with open(ors_path) as in_file:
        fc = json.load(in_file)
    for feature in fc.get("features", []):
        _add_feature(shape(feature["geometry"]), ORIGIN_ADDRESS,
                     feature["properties"].get("profile"), "ors",
                     feature["properties"].get("minutes"))

# Module 2 Mapbox.
mb_path = os.path.join(OUTPUT_FOLDER, "isochrones", "mapbox_raw.geojson")
if os.path.exists(mb_path):
    with open(mb_path) as in_file:
        fc = json.load(in_file)
    for feature in fc.get("features", []):
        _add_feature(shape(feature["geometry"]), ORIGIN_ADDRESS,
                     feature["properties"].get("profile"), "mapbox",
                     feature["properties"].get("minutes"))

# Module 5 comparison walksheds.
try:
    for address, polygon in comparison_walksheds.items():
        _add_feature(polygon, address, "foot-walking", "ors", 15)
except NameError:
    pass

# Module 6 student isochrones.
explore_path = os.path.join(OUTPUT_FOLDER, "exports", "exploration_isochrones.geojson")
if os.path.exists(explore_path):
    with open(explore_path) as in_file:
        fc = json.load(in_file)
    for feature in fc.get("features", []):
        _add_feature(shape(feature["geometry"]),
                     feature["properties"].get("origin"),
                     feature["properties"].get("mode"), "ors",
                     feature["properties"].get("minutes"))

bundle_path = os.path.join(OUTPUT_FOLDER, "exports", "all_isochrones.geojson")
with open(bundle_path, "w") as out_file:
    json.dump({"type":"FeatureCollection", "features": all_features}, out_file)
print(f"Bundled {len(all_features)} isochrones into: {bundle_path}")

In [ ]:
# Step 2: copy accessibility + time-of-day CSVs into the exports folder if present.
import shutil
for src, dst in [
    (os.path.join(OUTPUT_FOLDER, "accessibility", "access_score.csv"),
     os.path.join(OUTPUT_FOLDER, "exports", "accessibility_scores.csv")),
    (os.path.join(OUTPUT_FOLDER, "timeofday", "time_of_day_comparison.csv"),
     os.path.join(OUTPUT_FOLDER, "exports", "time_of_day.csv")),
]:
    if os.path.exists(src):
        shutil.copyfile(src, dst)
        print(f"Copied: {src} -> {dst}")
    else:
        print(f"(skipped) {src} does not exist yet")

In [ ]:
# Step 3: final layered summary folium map.
import requests
from shapely.geometry import shape as _shape

center = (ORIGIN_LATLNG if ORIGIN_LATLNG is not None
          else (40.713, -74.006))
final_map = folium.Map(location=list(center), zoom_start=11, tiles="cartodbpositron")

# Layer 1 — comparison isochrones (Module 5) as toggleable groups.
try:
    palette = ["#3C4ED6", "#C0392B", "#16A085", "#E67E22", "#7B241C"]
    for index, (address, polygon) in enumerate(comparison_walksheds.items()):
        color = palette[index % len(palette)]
        group = folium.FeatureGroup(name=f"Comparison: {address.split(',')[0]}",
                                    show=False)
        folium.GeoJson(polygon.__geo_interface__,
                       style_function=lambda feature, c=color: {"fillColor": c,
                                                                 "color": c,
                                                                 "weight": 1.5,
                                                                 "fillOpacity": 0.30},
                       tooltip=address).add_to(group)
        group.add_to(final_map)
except NameError:
    pass

# Layer 2 — student's exploration isochrones.
if os.path.exists(explore_path):
    with open(explore_path) as in_file:
        student_fc = json.load(in_file)
    student_group = folium.FeatureGroup(name="My exploration (Module 6)", show=True)
    for feature in student_fc.get("features", []):
        folium.GeoJson(feature["geometry"],
                       style_function=lambda f: {"fillColor": "#9B59B6",
                                                  "color": "#1B1B33",
                                                  "weight": 2,
                                                  "fillOpacity": 0.30},
                       tooltip=feature["properties"].get("origin",
                                                          "(my origin)")).add_to(student_group)
    student_group.add_to(final_map)

# Layer 3 — destination points (Module 4) as toggleable overlays.
try:
    DEST_COLORS_FINAL = {"parks":"#117864","public_schools":"#3C4ED6",
                         "libraries":"#7B241C","health_facilities":"#C0392B",
                         "retail_grocery":"#E67E22","subway_entrances":"#1B1B33"}
    for label, point_gdf in destinations.items():
        if point_gdf.empty: continue
        group = folium.FeatureGroup(name=f"All {label}", show=False)
        # Sample at most 500 points per type so the map stays responsive.
        sample = point_gdf.sample(min(500, len(point_gdf)), random_state=42)
        for _, row in sample.iterrows():
            folium.CircleMarker(
                location=[row.geometry.y, row.geometry.x],
                radius=2, color=DEST_COLORS_FINAL[label],
                fill=True, fill_opacity=0.7).add_to(group)
        group.add_to(final_map)
except NameError:
    pass

# Layer 4 — FEMA-ish current 100-year floodplain (Future Floodplain 2020s; the
# spec's inra-ujwb is 404, this is the current NYC-published layer).
try:
    flood_response = requests.get(
        "https://data.cityofnewyork.us/resource/aqw3-vugz.json",
        params={"$limit": "5000"}, timeout=60)
    flood_rows = flood_response.json()
    flood_features = []
    for row in flood_rows:
        raw_geom = row.get("the_geom")
        if isinstance(raw_geom, dict):
            flood_features.append({"type": "Feature",
                                   "geometry": raw_geom,
                                   "properties": {"fld_zone": row.get("fld_zone")}})
    if flood_features:
        folium.GeoJson({"type":"FeatureCollection","features":flood_features},
                       style_function=lambda feature: {"fillColor": "#3C4ED6",
                                                        "color": "#3C4ED6",
                                                        "weight": 0.4,
                                                        "fillOpacity": 0.20},
                       name="100-year floodplain (current)",
                       show=False).add_to(final_map)
except Exception as flood_error:
    print(f"Could not fetch floodplain layer: {flood_error}")

folium.LayerControl(collapsed=False).add_to(final_map)
final_path = os.path.join(OUTPUT_FOLDER, "maps", "final_summary_map.html")
final_map.save(final_path)
print(f"Saved: {final_path}")
display(final_map)

---

## Closing

Isochrone maps are persuasive. A well-made isochrone comparison can make transit inequality visible in a way that a data table cannot. **That persuasive power is also a responsibility.**

Before you use these maps in a presentation or a report, sit with these questions:

- **Are the APIs you used equally accurate across all neighborhoods you analyzed?** OpenStreetMap data quality is lower in some outer-borough areas — pedestrian tagging in particular. A walkshed that looks small in those areas may be a real signal, or it may be a data gap. Often both.

- **Does the routing model account for real-world barriers that do not appear in street-network data?** Fear, cost, disability, caregiving responsibilities, undocumented status — none of these are in OSM, in PLUTO, in GTFS, in any of the datasets this notebook uses.

- **Who is the audience for this map, and how might they misread it?** A good map prevents misreadings by what it leaves out as much as by what it puts in. A 15-minute isochrone with no walkability or safety overlay is a *claim about geometry*, not a claim about lived experience — be careful that your audience reads it that way.

- **What would it mean to show this map to the residents of the lowest-access neighborhood and ask them to respond to it?** If the answer is "they would say I missed the most important thing," the map needs another iteration before it is ready to leave the notebook.

The isochrone is a starting point for a conversation, not the conclusion of one.
